**Hyperactivation is a distinct change in sperm motility** characterized by:

* High-amplitude, asymmetrical flagellar beating.

* Increased head movement.

* Non-linear, often erratic or circular trajectories.

In [ ]:
import numpy as np
import pandas as pd
import glob
import os
import plotly.express as px

import scipy.signal
from scipy.signal import savgol_filter
from scipy.stats import entropy as shannon_entropy
import itertools

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d

from matplotlib import pyplot as plt
from scipy.fft import fft, fftfreq

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.colors import qualitative
import plotly.io as pio
import csv
import json
from IPython.display import Video

# My functions

In [ ]:
def interpolFFT(frecs,A,t):
    n=len(A)
    acum=0
    for k in range(n):
        acum=acum+A[k]*np.exp(2*np.pi*1j*t*frecs[k])
    return acum/n

def Dfourfit(recf,reca,t):# Primera derivada de la función dada
    acum=0
    n=len(reca)
    for i in range(n):
        acum=acum+reca[i]*recf[i]*np.exp(2*1j*np.pi*recf[i]*t)
    return 2*np.pi*1j*acum/(90*n)

def D2fourfit(recf,reca,t):# Segunda derivada de la función dada
  acum=0
  for i in range(len(recf)):
    acum=acum+reca[i]*recf[i]**2*np.exp(2*1j*np.pi*recf[i]*t)
  return -4*np.pi**2*acum/(90**2*len(recf))

def dist2D(x1,y1,x2,y2):
    xdifsq=(x2-x1)**2
    ydifsq=(y2-y1)**2
    return(np.sqrt(xdifsq+ydifsq))

def dist3D(x1,y1,z1,x2,y2,z2):
    xdifsq=(x2-x1)**2
    ydifsq=(y2-y1)**2
    zdifsq=(z2-z1)**2
    return(np.sqrt(xdifsq+ydifsq+zdifsq))

def curvature_torsion(x, y, z):
    r = np.vstack((x, y, z)).T
    dr = np.gradient(r, axis=0)
    d2r = np.gradient(dr, axis=0)
    d3r = np.gradient(d2r, axis=0)

    cross = np.cross(dr, d2r)
    norm_cross = np.linalg.norm(cross, axis=1)
    norm_dr = np.linalg.norm(dr, axis=1)
    curvature = norm_cross / (norm_dr**3 + 1e-8)

    torsion = np.einsum('ij,ij->i', cross, d3r) / (norm_cross**2 + 1e-8)
    return curvature, torsion

def curvDist3D(curva,nptos=400):
    lcl=0.0
    for pto in range(nptos-1):                               
        xi=curva.x.iloc[pto]
        yi=curva.y.iloc[pto]
        zi=curva.z.iloc[pto]
        xf=curva.x.iloc[pto+1]
        yf=curva.y.iloc[pto+1]
        zf=curva.z.iloc[pto+1]
        lcl=lcl+dist3D(xi,yi,zi,xf,yf,zf)
    return lcl

def conjuntoDiametro3D(curva, nptos=400):
    lstLongitudes=[]
    xi=curva.x.iloc[0]
    yi=curva.y.iloc[0]
    zi=curva.z.iloc[0]

    for pto in range(nptos-1):
        xf=curva.x.iloc[pto]
        yf=curva.y.iloc[pto]
        zf=curva.z.iloc[pto]
        lstLongitudes.append(dist3D(xi,yi,zi,xf,yf,zf))
    d=np.max(lstLongitudes)
    return d

def homogenizarRangos(curva1,curva2):
    xmax1=curva1.x.max()
    xmin1=curva1.x.min()
    ymax1=curva1.y.max()
    ymin1=curva1.y.min()
    zmax1=curva1.z.max()
    zmin1=curva1.z.min()
    xmax2=curva2.x.max()
    xmin2=curva2.x.min()
    ymax2=curva2.y.max()
    ymin2=curva2.y.min()
    zmax2=curva2.z.max()
    zmin2=curva2.z.min()

    xmax=np.max([xmax1,xmax2])
    ymax=np.max([ymax1,ymax2])
    zmax=np.max([zmax1,zmax2])
    xmin=np.min([xmin1,xmin2])
    ymin=np.min([ymin1,ymin2])
    zmin=np.min([zmin1,zmin2])

    
    deltax=xmax-xmin
    deltay=ymax-ymin
    deltaz=zmax-zmin
    maxdelta=np.max([deltax,deltay,deltaz])
    xadd=maxdelta-deltax
    yadd=maxdelta-deltay
    zadd=maxdelta-deltaz
    xadd=np.ceil(xadd/2)
    yadd=np.ceil(yadd/2)
    zadd=np.ceil(zadd/2)
    xmax=xmax+xadd+5
    xmin=xmin-xadd-5
    ymax=ymax+yadd+5
    ymin=ymin-yadd-5
    zmax=zmax+zadd+5
    zmin=zmin-zadd-5
    return [xmin,xmax,ymin,ymax,zmin,zmax]

def dimFrac3D(curva, nptos=400):
    lcl=curvDist3D(curva)
    d=conjuntoDiametro3D(curva)
    curveDimFractal=np.log10(nptos-1)/(np.log10(nptos-1)+np.log10(d/lcl))
    return curveDimFractal

def reflejarCurva3D(curva):
    micurva=curva.copy()
    x0=micurva.x.iloc[0]
    y0=micurva.y.iloc[0]
    z0=micurva.z.iloc[0]
    micurva.x=micurva.x-x0
    micurva.y=micurva.y-y0
    micurva.z=micurva.z-z0

    micurva.y=-micurva.y         # se refleja 

    micurva.x=micurva.x+x0
    micurva.y=micurva.y+y0
    micurva.z=micurva.z+z0
    return micurva

def buscarSwcidx(cel):
    nswcs=identificadoresArchivosSwcs.shape[0]
    fecha=cel.split('_')[-2]
    exp=cel.split('_')[-1]
    fileid='No encontrada'
    print('Buscando: ',fecha,exp)
    for j in range(nswcs):
        if(fecha in identificadoresArchivosSwcs.swc.iloc[j]):
            xp=identificadoresArchivosSwcs.swc.iloc[j].split('_')[-2]
            if(exp == xp):
                fileid=identificadoresArchivosSwcs.fileID.iloc[j].strip("'")
                print('Encontrado: ',identificadoresArchivosSwcs.swc.iloc[j].strip("'"))
                print(cel, ' corresponde a ')
                print('FileID= ', fileid,'\n')
    # return fileid
    encontrada='a'
    norigs=len(curvasFlagelares)
    for i in range(norigs):
        if(fileid in curvasFlagelares[i][0].nom.iloc[0]):
            encontrada=curvasFlagelares[i][0].nom.iloc[0]
            print('Orig: ',i,encontrada)
    if( encontrada != 'a'):
        nzenodo=len(curvasFlagelaresZenodo)
        for i in range(nzenodo):
            if(cel in curvasFlagelaresZenodo[i][0]):
                print('Zen: ',i,curvasFlagelaresZenodo[i][0])
    else:
        print('No encontrada')

def reflejarCurva3D(curva):
    micurva=curva.copy()
    x0=micurva.x.iloc[0]
    y0=micurva.y.iloc[0]
    z0=micurva.z.iloc[0]
    micurva.x=micurva.x-x0
    micurva.y=micurva.y-y0
    micurva.z=micurva.z-z0

    micurva.y=-micurva.y         # se refleja 

    micurva.x=micurva.x+x0
    micurva.y=micurva.y+y0
    micurva.z=micurva.z+z0
    return micurva

# Preprocessing

def preprocess(X, smooth=True):
    """
    X: (T, D) multivariate time-series
    """
    X = np.asarray(X, dtype=float)
    
    # z-score normalization
    X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-9)
    
    if smooth:
        X = savgol_filter(X, window_length=11, polyorder=2, axis=0)
    
    return X


# Kinematics

def kinematics(X, dt=1/90):
    """
    Returns velocity and acceleration
    """
    V = np.gradient(X, dt, axis=0)
    A = np.gradient(V, dt, axis=0)
    return V, A


# Dynamical Descriptors

def curvature(V, A):
    """
    Curvature for vector-valued motion
    """
    cross = np.cross(V, A)
    num = np.linalg.norm(cross, axis=1)
    den = (np.linalg.norm(V, axis=1) ** 3) + 1e-9
    return num / den


def jerk(A, dt=1/90):
    """
    Time derivative of acceleration magnitude
    """
    a_mag = np.linalg.norm(A, axis=1)
    J = np.gradient(a_mag, dt)
    return J


def signal_entropy(signal, bins=30):
    hist, _ = np.histogram(signal, bins=bins, density=True)
    return shannon_entropy(hist + 1e-12)


def nonlinearity_measure(signal):
    """
    Measures deviation from linear dynamics
    """
    return np.mean(np.abs(np.diff(signal, n=2)))


def asymmetry_measure(X):
    """
    Generic asymmetry between dimensions
    """
    D = X.shape[1]
    diffs = []
    for i in range(D):
        for j in range(i+1, D):
            diffs.append(np.mean(np.abs(X[:,i] - X[:,j])))
    return np.mean(diffs)


def periodicity_measure(signal):
    """
    Harmonicity / periodic structure indicator
    """
    fft = np.fft.rfft(signal)
    power = np.abs(fft)**2
    power /= power.sum() + 1e-12
    return power.max()   # high → periodic, low → chaotic


# Main Descriptor Function

def compute_dynamical_descriptors(X, dt=1/90):
    """
    X shape: (T, D)
    returns: feature vector + dictionary
    """
    X = preprocess(X)
    V, A = kinematics(X, dt)

    speed = np.linalg.norm(V, axis=1)
    accel = np.linalg.norm(A, axis=1)

    kappa = curvature(V, A)
    J = jerk(A, dt)

    features = {
        # Kinematics
        "mean_speed": speed.mean(),
        "std_speed": speed.std(),
        "max_speed": speed.max(),

        "mean_accel": accel.mean(),
        "std_accel": accel.std(),
        "max_accel": accel.max(),

        # Geometry
        "mean_curvature": np.nanmean(kappa),
        "std_curvature": np.nanstd(kappa),
        "max_curvature": np.nanmax(kappa),

        # Dynamics
        "mean_jerk": J.mean(),
        "std_jerk": J.std(),

        # Complexity
        "speed_entropy": signal_entropy(speed),
        "accel_entropy": signal_entropy(accel),

        "speed_nonlinearity": nonlinearity_measure(speed),
        "accel_nonlinearity": nonlinearity_measure(accel),

        # Structure
        "asymmetry": asymmetry_measure(X),
        "periodicity": periodicity_measure(speed),

        # Energy-like
        "kinetic_energy": np.mean(speed**2)
    }

    feature_vector = np.array(list(features.values()))
    return feature_vector, features


# Batch Processing

def build_feature_matrix(dataset, dt=1/90):
    """
    dataset: list or array of shape (N, T, D)
    """
    F = []
    dicts = []
    for Xi in dataset:
        fv, fd = compute_dynamical_descriptors(Xi, dt)
        F.append(fv)
        dicts.append(fd)
    return np.array(F), dicts

def descri(arreglo):
    mini=np.argmin(arreglo)
    maxi=np.argmax(arreglo)
    print('Min= ',np.min(arreglo),', idx= ',mini)
    print('Max= ',np.max(arreglo),', idx= ',maxi)
    temp=[]
    mu=np.mean(arreglo)
    for i in range(len(arreglo)):
        temp.append(np.abs(arreglo[i]-mu))
    meanClosest=np.argmin(temp)
    print('Mean= ',mu,', idxClosest= ',meanClosest)

    return mini,maxi,meanClosest

# def plot_single_cell_flagella_with_head(
#     flagella_data,
#     cell_index=0,
#     x_col="x",
#     y_col="y",
#     z_col="z",
#     title=None,
#     line_width=4,
#     marker_size=3,
#     head_marker_size=10,
#     show_markers=False
# ):
#     """
#     Animate a single cell flagellum with a larger point
#     marking the initial point of the curve.

#     Parameters
#     ----------
#     head_marker_size : int
#         Size of the initial point marker
#     """

#     # ----------------------------------------
#     # Detect input type
#     # ----------------------------------------

#     if isinstance(flagella_data[0], list):

#         cell_data = flagella_data[cell_index]

#     else:

#         cell_data = flagella_data

#     n_times = len(cell_data)

#     if title is None:
#         title = f"3D Flagella Animation — Cell {cell_index}"

#     # ----------------------------------------
#     # Compute global axis limits
#     # ----------------------------------------

#     all_x, all_y, all_z = [], [], []

#     for df in cell_data:

#         all_x.extend(df[x_col].values)
#         all_y.extend(df[y_col].values)
#         all_z.extend(df[z_col].values)

#     x_range = [min(all_x), max(all_x)]
#     y_range = [min(all_y), max(all_y)]
#     z_range = [min(all_z), max(all_z)]

#     # ----------------------------------------
#     # Initial frame
#     # ----------------------------------------

#     fig = go.Figure()
#     fig.update_layout(autosize=False,
#         width=1500,
#         height=1000,
#         scene_aspectmode='cube'
#         # margin=dict(l=10, r=10, t=10, b=10, pad=10)
#         )
#     df0 = cell_data[0]

#     # Flagellum line
#     fig.add_trace(
#         go.Scatter3d(
#             x=df0[x_col],
#             y=df0[y_col],
#             z=df0[z_col],
#             mode="lines+markers" if show_markers else "lines",
#             line=dict(width=line_width),
#             marker=dict(size=marker_size),
#             name="Flagellum"
#         )
#     )

#     # Initial point marker (HEAD / BASE)
#     fig.add_trace(
#         go.Scatter3d(
#             x=[df0[x_col].iloc[0]],
#             y=[df0[y_col].iloc[0]],
#             z=[df0[z_col].iloc[0]],
#             mode="markers",
#             marker=dict(
#                 size=head_marker_size,
#                 symbol="circle"
#             ),
#             name="Initial point"
#         )
#     )

#     # ----------------------------------------
#     # Frames
#     # ----------------------------------------

#     frames = []

#     for t in range(n_times):

#         df = cell_data[t]

#         frames.append(
#             go.Frame(
#                 data=[

#                     # Flagellum
#                     go.Scatter3d(
#                         x=df[x_col],
#                         y=df[y_col],
#                         z=df[z_col],
#                         mode="lines+markers" if show_markers else "lines",
#                         line=dict(width=line_width),
#                         marker=dict(size=marker_size),
#                     ),

#                     # Initial point
#                     go.Scatter3d(
#                         x=[df[x_col].iloc[0]],
#                         y=[df[y_col].iloc[0]],
#                         z=[df[z_col].iloc[0]],
#                         mode="markers",
#                         marker=dict(
#                             size=head_marker_size
#                         ),
#                     )

#                 ],
#                 name=str(t)
#             )
#         )

#     fig.frames = frames

#     # ----------------------------------------
#     # Slider
#     # ----------------------------------------

#     slider_steps = []

#     for t in range(n_times):

#         slider_steps.append(
#             dict(
#                 method="animate",
#                 args=[
#                     [str(t)],
#                     dict(
#                         mode="immediate",
#                         frame=dict(duration=40, redraw=True),
#                         transition=dict(duration=0)
#                     )
#                 ],
#                 label=str(t)
#             )
#         )

#     sliders = [
#         dict(
#             active=0,
#             pad={"t": 50},
#             steps=slider_steps
#         )
#     ]

#     # ----------------------------------------
#     # Layout
#     # ----------------------------------------

#     fig.update_layout(
#         title=title,

#         scene=dict(
#             xaxis=dict(range=x_range),
#             yaxis=dict(range=y_range),
#             zaxis=dict(range=z_range),
#             aspectmode="data"
#         ),

#         sliders=sliders,

#         updatemenus=[
#             dict(
#                 type="buttons",
#                 showactive=False,
#                 buttons=[

#                     dict(
#                         label="Play",
#                         method="animate",
#                         args=[
#                             None,
#                             dict(
#                                 frame=dict(duration=40, redraw=True),
#                                 transition=dict(duration=0),
#                                 fromcurrent=True
#                             )
#                         ]
#                     ),

#                     dict(
#                         label="Pause",
#                         method="animate",
#                         args=[
#                             [None],
#                             dict(
#                                 frame=dict(duration=0),
#                                 transition=dict(duration=0),
#                                 mode="immediate"
#                             )
#                         ]
#                     )

#                 ]
#             )
#         ]
#     )

#     return fig



def compute_curvature(df, x_col="x", y_col="y", z_col="z", smooth_sigma=1):
    x = df[x_col].values
    y = df[y_col].values
    z = df[z_col].values

    # Optional smoothing (VERY important for curvature stability)
    if smooth_sigma is not None:
        x = gaussian_filter1d(x, smooth_sigma)
        y = gaussian_filter1d(y, smooth_sigma)
        z = gaussian_filter1d(z, smooth_sigma)

    r = np.stack([x, y, z], axis=1)

    dr = np.gradient(r, axis=0)
    d2r = np.gradient(dr, axis=0)

    cross = np.cross(dr, d2r)

    num = np.linalg.norm(cross, axis=1)
    den = np.linalg.norm(dr, axis=1)**3 + 1e-8

    return num / den


def animate_flagella_with_feature_vertical_curvature(
    cell_data,
    feature,
    time=None,
    x_col="x",
    y_col="y",
    z_col="z",
    feature_name="Feature",
    head_marker_size=10,
    frame_duration=80,
    peak_prominence=0.1,
    peak_distance=5,
    max_peak_buttons=12,
    smooth_sigma=1
):
    """
    FULL visualization:

    TOP:
        3D flagellum colored by curvature

    BOTTOM:
        time series + peaks + moving point

    Includes:
        - peak navigation
        - step controls
        - consistent curvature scaling
    """

    feature = np.asarray(feature)
    T = len(cell_data)

    if time is None:
        time = np.arange(T)
    time = np.asarray(time)

    # -------------------------------------------------
    # Peak detection
    # -------------------------------------------------

    peaks, _ = find_peaks(
        feature,
        prominence=peak_prominence,
        distance=peak_distance
    )

    # -------------------------------------------------
    # Curvature precomputation
    # -------------------------------------------------

    curvatures = [
        compute_curvature(df, x_col, y_col, z_col, smooth_sigma)
        for df in cell_data
    ]

    global_kmin = min(k.min() for k in curvatures)
    global_kmax = max(k.max() for k in curvatures)

    # Optional clipping (robust visualization)
    global_kmax = np.percentile(
        np.concatenate(curvatures), 99
    )

    # -------------------------------------------------
    # Axis limits
    # -------------------------------------------------

    all_x, all_y, all_z = [], [], []

    for df in cell_data:
        all_x.extend(df[x_col].values)
        all_y.extend(df[y_col].values)
        all_z.extend(df[z_col].values)

    x_range = [min(all_x), max(all_x)]
    y_range = [min(all_y), max(all_y)]
    z_range = [min(all_z), max(all_z)]

    # -------------------------------------------------
    # Subplots
    # -------------------------------------------------

    fig = make_subplots(
        rows=2,
        cols=1,
        row_heights=[0.65, 0.35],
        vertical_spacing=0.08,
        specs=[[{"type": "scene"}], [{"type": "xy"}]],
        subplot_titles=("3D Flagellum (Curvature)", feature_name)
    )

    # -------------------------------------------------
    # Initial frame
    # -------------------------------------------------

    df0 = cell_data[0]
    k0 = curvatures[0]

    fig.add_trace(
        go.Scatter3d(
            x=df0[x_col],
            y=df0[y_col],
            z=df0[z_col],
            mode="lines",
            line=dict(
                width=6,
                color=k0,
                colorscale="Viridis",
                cmin=global_kmin,
                cmax=global_kmax,
                colorbar=dict(title="Curvature")
            ),
            name="Curvature"
        ),
        row=1, col=1
    )

    # Head marker
    fig.add_trace(
        go.Scatter3d(
            x=[df0[x_col].iloc[0]],
            y=[df0[y_col].iloc[0]],
            z=[df0[z_col].iloc[0]],
            mode="markers",
            marker=dict(size=head_marker_size),
            name="Head"
        ),
        row=1, col=1
    )

    # Feature line
    fig.add_trace(
        go.Scatter(
            x=time,
            y=feature,
            mode="lines+markers",
            name=feature_name
        ),
        row=2, col=1
    )

    # Moving point
    fig.add_trace(
        go.Scatter(
            x=[time[0]],
            y=[feature[0]],
            mode="markers",
            marker=dict(size=12),
            name="Current"
        ),
        row=2, col=1
    )

    # Peaks
    fig.add_trace(
        go.Scatter(
            x=time[peaks],
            y=feature[peaks],
            mode="markers",
            marker=dict(size=10, symbol="diamond"),
            name="Peaks"
        ),
        row=2, col=1
    )

    # -------------------------------------------------
    # Frames
    # -------------------------------------------------

    frames = []

    for t in range(T):

        df = cell_data[t]
        kappa = curvatures[t]

        frames.append(
            go.Frame(
                data=[

                    # Curvature-colored flagellum
                    go.Scatter3d(
                        x=df[x_col],
                        y=df[y_col],
                        z=df[z_col],
                        mode="lines",
                        line=dict(
                            width=6,
                            color=kappa,
                            colorscale="Viridis",
                            cmin=global_kmin,
                            cmax=global_kmax
                        ),
                        showlegend=False
                    ),

                    # Head
                    go.Scatter3d(
                        x=[df[x_col].iloc[0]],
                        y=[df[y_col].iloc[0]],
                        z=[df[z_col].iloc[0]],
                        mode="markers",
                        marker=dict(size=head_marker_size),
                        showlegend=False
                    ),

                    # Feature line
                    go.Scatter(
                        x=time,
                        y=feature,
                        mode="lines+markers"
                    ),

                    # Moving point
                    go.Scatter(
                        x=[time[t]],
                        y=[feature[t]],
                        mode="markers",
                        marker=dict(size=12)
                    ),

                    # Peaks
                    go.Scatter(
                        x=time[peaks],
                        y=feature[peaks],
                        mode="markers",
                        marker=dict(size=10, symbol="diamond")
                    )

                ],
                name=str(t)
            )
        )

    fig.frames = frames

    # -------------------------------------------------
    # Slider
    # -------------------------------------------------

    steps = [
        dict(
            method="animate",
            args=[
                [str(t)],
                dict(
                    mode="immediate",
                    frame=dict(duration=frame_duration, redraw=True),
                    transition=dict(duration=0)
                )
            ],
            label=str(t)
        )
        for t in range(T)
    ]

    sliders = [dict(active=0, pad={"t": 50}, steps=steps)]

    # -------------------------------------------------
    # Peak buttons
    # -------------------------------------------------

    peak_buttons = [
        dict(
            label=f"Peak {i}",
            method="animate",
            args=[
                [str(p)],
                dict(
                    mode="immediate",
                    frame=dict(duration=0, redraw=True),
                    transition=dict(duration=0)
                )
            ]
        )
        for i, p in enumerate(peaks[:max_peak_buttons])
    ]

    # -------------------------------------------------
    # Layout
    # -------------------------------------------------

    fig.update_layout(
        height=800,

        scene=dict(
            xaxis=dict(range=x_range),
            yaxis=dict(range=y_range),
            zaxis=dict(range=z_range),
            aspectmode="data"
        ),

        xaxis=dict(title="Time"),
        yaxis=dict(title=feature_name, range=[feature.min(), feature.max()]),

        sliders=sliders,

        updatemenus=[

            dict(
                type="buttons",
                showactive=False,
                direction="left",
                x=0.1,
                y=1.15,
                buttons=[

                    dict(
                        label="⏮ Prev",
                        method="animate",
                        args=[None, dict(mode="immediate",
                                         frame=dict(duration=0),
                                         transition=dict(duration=0),
                                         direction="reverse")]
                    ),

                    dict(
                        label="Play",
                        method="animate",
                        args=[None, dict(frame=dict(duration=frame_duration),
                                         transition=dict(duration=0),
                                         fromcurrent=True)]
                    ),

                    dict(
                        label="Pause",
                        method="animate",
                        args=[[None], dict(frame=dict(duration=0),
                                           transition=dict(duration=0),
                                           mode="immediate")]
                    ),

                    dict(
                        label="⏭ Next",
                        method="animate",
                        args=[None, dict(mode="immediate",
                                         frame=dict(duration=0),
                                         transition=dict(duration=0))]
                    )

                ]
            ),

            dict(
                type="dropdown",
                x=0.6,
                y=1.15,
                buttons=peak_buttons,
                showactive=True
            )

        ]
    )

    return fig



# =========================================================
# TORSION COMPUTATION
# =========================================================

def compute_torsion(
    df,
    x_col="x",
    y_col="y",
    z_col="z",
    smooth_sigma=1
):
    """
    Compute torsion of a 3D curve.

    τ = ((r' × r'') · r''') / ||r' × r''||²
    """

    x = df[x_col].values
    y = df[y_col].values
    z = df[z_col].values

    # -------------------------------------
    # Optional smoothing
    # -------------------------------------

    if smooth_sigma is not None:

        x = gaussian_filter1d(x, smooth_sigma)
        y = gaussian_filter1d(y, smooth_sigma)
        z = gaussian_filter1d(z, smooth_sigma)

    r = np.stack([x, y, z], axis=1)

    # First derivative
    dr = np.gradient(r, axis=0)

    # Second derivative
    d2r = np.gradient(dr, axis=0)

    # Third derivative
    d3r = np.gradient(d2r, axis=0)

    # Cross product
    cross = np.cross(dr, d2r)

    numerator = np.sum(cross * d3r, axis=1)

    denominator = (
        np.linalg.norm(cross, axis=1)**2 + 1e-8
    )

    torsion = numerator / denominator

    return torsion


# =========================================================
# MAIN ANIMATION FUNCTION
# =========================================================

def animate_flagella_with_feature_vertical_torsion(
    cell_data,
    feature,
    time=None,
    x_col="x",
    y_col="y",
    z_col="z",
    feature_name="Feature",
    head_marker_size=10,
    frame_duration=80,
    peak_prominence=0.1,
    peak_distance=5,
    max_peak_buttons=12,
    smooth_sigma=1,
    torsion_clip_percentile=99,
    escalaColor="Jet"
):
    """
    FULL visualization:

    TOP:
        3D flagellum colored by torsion

    BOTTOM:
        time series + peaks + moving point

    Includes:
        - peak navigation
        - slider
        - play/pause
        - step controls
        - torsion coloring
    """

    feature = np.asarray(feature)

    T = len(cell_data)

    if time is None:
        time = np.arange(T)

    time = np.asarray(time)

    # =====================================================
    # PEAK DETECTION
    # =====================================================

    peaks, _ = find_peaks(
        feature,
        prominence=peak_prominence,
        distance=peak_distance
    )

    # =====================================================
    # PRECOMPUTE TORSION
    # =====================================================

    torsions = [
        compute_torsion(
            df,
            x_col,
            y_col,
            z_col,
            smooth_sigma
        )
        for df in cell_data
    ]

    # Global scaling
    all_torsion = np.concatenate(torsions)

    # Robust clipping
    limit = np.percentile(
        np.abs(all_torsion),
        torsion_clip_percentile
    )

    global_tmin = -limit
    global_tmax = limit

    # =====================================================
    # 3D AXIS LIMITS
    # =====================================================

    all_x, all_y, all_z = [], [], []

    for df in cell_data:

        all_x.extend(df[x_col].values)
        all_y.extend(df[y_col].values)
        all_z.extend(df[z_col].values)

    x_range = [min(all_x), max(all_x)]
    y_range = [min(all_y), max(all_y)]
    z_range = [min(all_z), max(all_z)]

    # =====================================================
    # CREATE SUBPLOTS
    # =====================================================

    fig = make_subplots(
        rows=2,
        cols=1,
        row_heights=[0.65, 0.35],
        vertical_spacing=0.08,
        specs=[
            [{"type": "scene"}],
            [{"type": "xy"}]
        ],
        subplot_titles=(
            "3D Flagellum (Torsion)",
            feature_name
        )
    )

    # =====================================================
    # INITIAL FRAME
    # =====================================================

    df0 = cell_data[0]
    t0 = torsions[0]

    # -------------------------------------
    # Torsion-colored flagellum
    # -------------------------------------

    fig.add_trace(
        go.Scatter3d(
            x=df0[x_col],
            y=df0[y_col],
            z=df0[z_col],
            mode="lines",

            line=dict(
                width=6,
                color=t0,
                colorscale=escalaColor,
                cmin=global_tmin,
                cmax=global_tmax,

                colorbar=dict(
                    title="Torsion"
                )
            ),

            name="Torsion"
        ),
        row=1,
        col=1
    )

    # -------------------------------------
    # Head marker
    # -------------------------------------

    fig.add_trace(
        go.Scatter3d(
            x=[df0[x_col].iloc[0]],
            y=[df0[y_col].iloc[0]],
            z=[df0[z_col].iloc[0]],

            mode="markers",

            marker=dict(
                size=head_marker_size
            ),

            name="Head"
        ),
        row=1,
        col=1
    )

    # -------------------------------------
    # Feature line
    # -------------------------------------

    fig.add_trace(
        go.Scatter(
            x=time,
            y=feature,
            mode="lines",
            name=feature_name
        ),
        row=2,
        col=1
    )

    # -------------------------------------
    # Moving point
    # -------------------------------------

    fig.add_trace(
        go.Scatter(
            x=[time[0]],
            y=[feature[0]],
            mode="markers",

            marker=dict(size=12),

            name="Current"
        ),
        row=2,
        col=1
    )

    # -------------------------------------
    # Peaks
    # -------------------------------------

    fig.add_trace(
        go.Scatter(
            x=time[peaks],
            y=feature[peaks],

            mode="markers",

            marker=dict(
                size=10,
                symbol="diamond"
            ),

            name="Peaks"
        ),
        row=2,
        col=1
    )

    # =====================================================
    # FRAMES
    # =====================================================

    frames = []

    for t in range(T):

        df = cell_data[t]
        torsion = torsions[t]

        frames.append(

            go.Frame(
                data=[

                    # ---------------------------------
                    # Torsion-colored flagellum
                    # ---------------------------------

                    go.Scatter3d(
                        x=df[x_col],
                        y=df[y_col],
                        z=df[z_col],

                        mode="lines",

                        line=dict(
                            width=6,
                            color=torsion,
                            colorscale=escalaColor,
                            cmin=global_tmin,
                            cmax=global_tmax
                        )
                    ),

                    # ---------------------------------
                    # Head
                    # ---------------------------------

                    go.Scatter3d(
                        x=[df[x_col].iloc[0]],
                        y=[df[y_col].iloc[0]],
                        z=[df[z_col].iloc[0]],

                        mode="markers",

                        marker=dict(
                            size=head_marker_size
                        )
                    ),

                    # ---------------------------------
                    # Feature line
                    # ---------------------------------

                    go.Scatter(
                        x=time,
                        y=feature,
                        mode="lines"
                    ),

                    # ---------------------------------
                    # Moving point
                    # ---------------------------------

                    go.Scatter(
                        x=[time[t]],
                        y=[feature[t]],

                        mode="markers",

                        marker=dict(size=12)
                    ),

                    # ---------------------------------
                    # Peaks
                    # ---------------------------------

                    go.Scatter(
                        x=time[peaks],
                        y=feature[peaks],

                        mode="markers",

                        marker=dict(
                            size=10,
                            symbol="diamond"
                        )
                    )

                ],

                name=str(t)
            )
        )

    fig.frames = frames

    # =====================================================
    # SLIDER
    # =====================================================

    steps = []

    for t in range(T):

        steps.append(
            dict(
                method="animate",

                args=[
                    [str(t)],

                    dict(
                        mode="immediate",

                        frame=dict(
                            duration=frame_duration,
                            redraw=True
                        ),

                        transition=dict(duration=0)
                    )
                ],

                label=str(t)
            )
        )

    sliders = [
        dict(
            active=0,
            pad={"t": 50},
            steps=steps
        )
    ]

    # =====================================================
    # PEAK BUTTONS
    # =====================================================

    peak_buttons = []

    for i, p in enumerate(peaks[:max_peak_buttons]):

        peak_buttons.append(

            dict(
                label=f"Peak {i}",

                method="animate",

                args=[
                    [str(p)],

                    dict(
                        mode="immediate",

                        frame=dict(
                            duration=0,
                            redraw=True
                        ),

                        transition=dict(duration=0)
                    )
                ]
            )
        )

    # =====================================================
    # LAYOUT
    # =====================================================

    fig.update_layout(

        height=850,

        scene=dict(

            xaxis=dict(range=x_range),
            yaxis=dict(range=y_range),
            zaxis=dict(range=z_range),

            aspectmode="data"
        ),

        xaxis=dict(
            title="Time"
        ),

        yaxis=dict(
            title=feature_name,
            range=[feature.min(), feature.max()]
        ),

        sliders=sliders,

        updatemenus=[

            # =================================
            # MAIN CONTROLS
            # =================================

            dict(
                type="buttons",

                showactive=False,

                direction="left",

                x=0.1,
                y=1.15,

                buttons=[

                    dict(
                        label="⏮ Prev",

                        method="animate",

                        args=[
                            None,

                            dict(
                                mode="immediate",

                                frame=dict(duration=0),

                                transition=dict(duration=0),

                                direction="reverse"
                            )
                        ]
                    ),

                    dict(
                        label="Play",

                        method="animate",

                        args=[
                            None,

                            dict(
                                frame=dict(
                                    duration=frame_duration
                                ),

                                transition=dict(duration=0),

                                fromcurrent=True
                            )
                        ]
                    ),

                    dict(
                        label="Pause",

                        method="animate",

                        args=[
                            [None],

                            dict(
                                frame=dict(duration=0),

                                transition=dict(duration=0),

                                mode="immediate"
                            )
                        ]
                    ),

                    dict(
                        label="⏭ Next",

                        method="animate",

                        args=[
                            None,

                            dict(
                                mode="immediate",

                                frame=dict(duration=0),

                                transition=dict(duration=0)
                            )
                        ]
                    )

                ]
            ),

            # =================================
            # PEAK DROPDOWN
            # =================================

            dict(
                type="dropdown",

                x=0.6,
                y=1.15,

                buttons=peak_buttons,

                showactive=True
            )

        ]
    )

    return fig
    
# def animate_flagella_with_feature_vertical(
#     cell_data,
#     feature,
#     time=None,
#     x_col="x",
#     y_col="y",
#     z_col="z",
#     feature_name="Feature",
#     head_marker_size=10,
#     frame_duration=80,
#     peak_prominence=0.1,
#     peak_distance=5,
#     max_peak_buttons=12
# ):
#     """
#     Vertical synchronized animation with peak navigation.

#     TOP    -> 3D flagellum
#     BOTTOM -> time series + peaks

#     Includes:
#         - slider
#         - play/pause
#         - prev/next stepping
#         - jump to peaks dropdown
#     """

#     feature = np.asarray(feature)
#     T = len(cell_data)

#     if time is None:
#         time = np.arange(T)
#     time = np.asarray(time)

#     # -------------------------------------------------
#     # Peak detection
#     # -------------------------------------------------

#     peaks, _ = find_peaks(
#         feature,
#         prominence=peak_prominence,
#         distance=peak_distance
#     )

#     # -------------------------------------------------
#     # Compute 3D axis limits
#     # -------------------------------------------------

#     all_x, all_y, all_z = [], [], []

#     for df in cell_data:
#         all_x.extend(df[x_col].values)
#         all_y.extend(df[y_col].values)
#         all_z.extend(df[z_col].values)

#     x_range = [min(all_x), max(all_x)]
#     y_range = [min(all_y), max(all_y)]
#     z_range = [min(all_z), max(all_z)]

#     # -------------------------------------------------
#     # Create subplots
#     # -------------------------------------------------

#     fig = make_subplots(
#         rows=2,
#         cols=1,
#         row_heights=[0.65, 0.35],
#         vertical_spacing=0.08,
#         specs=[[{"type": "scene"}], [{"type": "xy"}]],
#         subplot_titles=("3D Flagellum", feature_name)
#     )

#     # -------------------------------------------------
#     # Initial frame
#     # -------------------------------------------------

#     df0 = cell_data[0]

#     # 3D flagellum
#     fig.add_trace(
#         go.Scatter3d(
#             x=df0[x_col],
#             y=df0[y_col],
#             z=df0[z_col],
#             mode="lines",
#             line=dict(width=4),
#             name="Flagellum"
#         ),
#         row=1, col=1
#     )

#     # Head marker
#     fig.add_trace(
#         go.Scatter3d(
#             x=[df0[x_col].iloc[0]],
#             y=[df0[y_col].iloc[0]],
#             z=[df0[z_col].iloc[0]],
#             mode="markers",
#             marker=dict(size=head_marker_size),
#             name="Head"
#         ),
#         row=1, col=1
#     )

#     # Feature line
#     fig.add_trace(
#         go.Scatter(
#             x=time,
#             y=feature,
#             mode="lines",
#             name=feature_name
#         ),
#         row=2, col=1
#     )

#     # Moving point
#     fig.add_trace(
#         go.Scatter(
#             x=[time[0]],
#             y=[feature[0]],
#             mode="markers",
#             marker=dict(size=12),
#             name="Current"
#         ),
#         row=2, col=1
#     )

#     # Peak markers
#     fig.add_trace(
#         go.Scatter(
#             x=time[peaks],
#             y=feature[peaks],
#             mode="markers",
#             marker=dict(size=10, symbol="diamond"),
#             name="Peaks"
#         ),
#         row=2, col=1
#     )

#     # -------------------------------------------------
#     # Frames
#     # -------------------------------------------------

#     frames = []

#     for t in range(T):

#         df = cell_data[t]

#         frames.append(
#             go.Frame(
#                 data=[

#                     # 3D line
#                     go.Scatter3d(
#                         x=df[x_col],
#                         y=df[y_col],
#                         z=df[z_col],
#                         mode="lines",
#                         line=dict(width=4)
#                     ),

#                     # Head
#                     go.Scatter3d(
#                         x=[df[x_col].iloc[0]],
#                         y=[df[y_col].iloc[0]],
#                         z=[df[z_col].iloc[0]],
#                         mode="markers",
#                         marker=dict(size=head_marker_size)
#                     ),

#                     # Feature line
#                     go.Scatter(
#                         x=time,
#                         y=feature,
#                         mode="lines"
#                     ),

#                     # Moving point
#                     go.Scatter(
#                         x=[time[t]],
#                         y=[feature[t]],
#                         mode="markers",
#                         marker=dict(size=12)
#                     ),

#                     # Peaks (static)
#                     go.Scatter(
#                         x=time[peaks],
#                         y=feature[peaks],
#                         mode="markers",
#                         marker=dict(size=10, symbol="diamond")
#                     )

#                 ],
#                 name=str(t)
#             )
#         )

#     fig.frames = frames

#     # -------------------------------------------------
#     # Slider
#     # -------------------------------------------------

#     steps = []

#     for t in range(T):
#         steps.append(
#             dict(
#                 method="animate",
#                 args=[
#                     [str(t)],
#                     dict(
#                         mode="immediate",
#                         frame=dict(duration=frame_duration, redraw=True),
#                         transition=dict(duration=0)
#                     )
#                 ],
#                 label=str(t)
#             )
#         )

#     sliders = [dict(active=0, pad={"t": 50}, steps=steps)]

#     # -------------------------------------------------
#     # Peak buttons
#     # -------------------------------------------------

#     peak_buttons = []

#     for i, p in enumerate(peaks[:max_peak_buttons]):
#         peak_buttons.append(
#             dict(
#                 label=f"Peak {i}",
#                 method="animate",
#                 args=[
#                     [str(p)],
#                     dict(
#                         mode="immediate",
#                         frame=dict(duration=0, redraw=True),
#                         transition=dict(duration=0)
#                     )
#                 ]
#             )
#         )

#     # -------------------------------------------------
#     # Layout + controls
#     # -------------------------------------------------

#     fig.update_layout(

#         height=800,

#         scene=dict(
#             xaxis=dict(range=x_range),
#             yaxis=dict(range=y_range),
#             zaxis=dict(range=z_range),
#             aspectmode="data"
#         ),

#         xaxis=dict(title="Time"),
#         yaxis=dict(title=feature_name, range=[feature.min(), feature.max()]),

#         sliders=sliders,

#         updatemenus=[

#             # Main controls
#             dict(
#                 type="buttons",
#                 showactive=False,
#                 direction="left",
#                 x=0.1,
#                 y=1.15,
#                 buttons=[

#                     dict(
#                         label="⏮ Prev",
#                         method="animate",
#                         args=[None, dict(mode="immediate",
#                                          frame=dict(duration=0),
#                                          transition=dict(duration=0),
#                                          direction="reverse")]
#                     ),

#                     dict(
#                         label="Play",
#                         method="animate",
#                         args=[None, dict(frame=dict(duration=frame_duration),
#                                          transition=dict(duration=0),
#                                          fromcurrent=True)]
#                     ),

#                     dict(
#                         label="Pause",
#                         method="animate",
#                         args=[[None], dict(frame=dict(duration=0),
#                                            transition=dict(duration=0),
#                                            mode="immediate")]
#                     ),

#                     dict(
#                         label="⏭ Next",
#                         method="animate",
#                         args=[None, dict(mode="immediate",
#                                          frame=dict(duration=0),
#                                          transition=dict(duration=0))]
#                     )

#                 ]
#             ),

#             # Peak dropdown
#             dict(
#                 type="dropdown",
#                 x=0.6,
#                 y=1.15,
#                 buttons=peak_buttons,
#                 showactive=True
#             )

#         ]
#     )

#     return fig

# def animate_flagella_with_feature_vertical(
#     cell_data,
#     feature,
#     time=None,
#     x_col="x",
#     y_col="y",
#     z_col="z",
#     feature_name="Feature",
#     head_marker_size=10,
#     frame_duration=40,
#     flagellarTitle="3D Flagellum"
# ):
#     """
#     Synchronized animation:

#         TOP    -> 3D flagellum
#         BOTTOM -> time series feature

#     Shared slider and animation timing.
#     """

#     feature = np.asarray(feature)

#     T = len(cell_data)

#     if time is None:
#         time = np.arange(T)

#     time = np.asarray(time)

#     # ----------------------------------
#     # Compute axis limits for 3D
#     # ----------------------------------

#     all_x, all_y, all_z = [], [], []

#     for df in cell_data:

#         all_x.extend(df[x_col].values)
#         all_y.extend(df[y_col].values)
#         all_z.extend(df[z_col].values)

#     x_range = [min(all_x), max(all_x)]
#     y_range = [min(all_y), max(all_y)]
#     z_range = [min(all_z), max(all_z)]

#     # ----------------------------------
#     # Create vertical subplots
#     # ----------------------------------

#     fig = make_subplots(
#         rows=2,
#         cols=1,
#         row_heights=[0.65, 0.35],
#         vertical_spacing=0.08,
#         specs=[
#             [{"type": "scene"}],
#             [{"type": "xy"}]
#         ],
#         subplot_titles=(
#             flagellarTitle,
#             feature_name
#         )
#     )

#     # ----------------------------------
#     # Initial frame
#     # ----------------------------------

#     df0 = cell_data[0]

#     # 3D flagellum

#     fig.add_trace(
#         go.Scatter3d(
#             x=df0[x_col],
#             y=df0[y_col],
#             z=df0[z_col],
#             mode="lines",
#             line=dict(width=4),
#             name="Flagellum"
#         ),
#         row=1,
#         col=1
#     )

#     # Head marker

#     fig.add_trace(
#         go.Scatter3d(
#             x=[df0[x_col].iloc[0]],
#             y=[df0[y_col].iloc[0]],
#             z=[df0[z_col].iloc[0]],
#             mode="markers",
#             marker=dict(size=head_marker_size),
#             name="Head"
#         ),
#         row=1,
#         col=1
#     )

#     # Feature line

#     fig.add_trace(
#         go.Scatter(
#             x=time,
#             y=feature,
#             mode="lines+markers",
#             name=feature_name
#         ),
#         row=2,
#         col=1
#     )

#     # Moving point

#     fig.add_trace(
#         go.Scatter(
#             x=[time[0]],
#             y=[feature[0]],
#             mode="markers",
#             marker=dict(size=12),
#             name="Current time"
#         ),
#         row=2,
#         col=1
#     )

#     # ----------------------------------
#     # Frames
#     # ----------------------------------

#     frames = []

#     for t in range(T):

#         df = cell_data[t]

#         frames.append(
#             go.Frame(
#                 data=[

#                     go.Scatter3d(
#                         x=df[x_col],
#                         y=df[y_col],
#                         z=df[z_col],
#                         mode="lines",
#                         line=dict(width=4)
#                     ),

#                     go.Scatter3d(
#                         x=[df[x_col].iloc[0]],
#                         y=[df[y_col].iloc[0]],
#                         z=[df[z_col].iloc[0]],
#                         mode="markers",
#                         marker=dict(size=head_marker_size)
#                     ),

#                     go.Scatter(
#                         x=time,
#                         y=feature,
#                         mode="lines+markers"
#                     ),

#                     go.Scatter(
#                         x=[time[t]],
#                         y=[feature[t]],
#                         mode="markers",
#                         marker=dict(size=12)
#                     )

#                 ],
#                 name=str(t)
#             )
#         )

#     fig.frames = frames

#     # ----------------------------------
#     # Slider
#     # ----------------------------------

#     steps = []

#     for t in range(T):

#         steps.append(
#             dict(
#                 method="animate",
#                 args=[
#                     [str(t)],
#                     dict(
#                         mode="immediate",
#                         frame=dict(duration=frame_duration, redraw=True),
#                         transition=dict(duration=0)
#                     )
#                 ],
#                 label=str(t)
#             )
#         )

#     sliders = [
#         dict(
#             active=0,
#             pad={"t": 50},
#             steps=steps
#         )
#     ]

#     # ----------------------------------
#     # Layout
#     # ----------------------------------

#     fig.update_layout(

#         height=800,

#         scene=dict(
#             xaxis=dict(range=x_range),
#             yaxis=dict(range=y_range),
#             zaxis=dict(range=z_range),
#             aspectmode="data"
#         ),

#         xaxis=dict(
#             title="Time"
#         ),

#         yaxis=dict(
#             title="Fractal Dimension",#feature_name,
#             range=[feature.min(), feature.max()]
#         ),

#         sliders=sliders,

#         updatemenus=[
#             dict(
#                 type="buttons",
#                 showactive=False,

#                 buttons=[

#                     dict(
#                         label="Play",
#                         method="animate",
#                         args=[
#                             None,
#                             dict(
#                                 frame=dict(duration=frame_duration, redraw=True),
#                                 transition=dict(duration=0),
#                                 fromcurrent=True
#                             )
#                         ]
#                     ),

#                     dict(
#                         label="Pause",
#                         method="animate",
#                         args=[
#                             [None],
#                             dict(
#                                 frame=dict(duration=0),
#                                 transition=dict(duration=0),
#                                 mode="immediate"
#                             )
#                         ]
#                     )

#                 ]
#             )
#         ]
#     )

#     return fig

# def animate_time_series_point(
#     feature,
#     time=None,
#     title="Time Series Animation",
#     y_label="Feature",
#     point_size=12,
#     frame_duration=40
# ):
#     """
#     Animate a moving point along a time series with slider.

#     Parameters
#     ----------
#     feature : array-like
#         1D time series

#     time : array-like, optional
#         Time vector. If None, uses indices.

#     point_size : int
#         Size of moving point

#     frame_duration : int
#         milliseconds between frames
#     """

#     feature = np.asarray(feature)

#     T = len(feature)

#     if time is None:
#         time = np.arange(T)

#     time = np.asarray(time)

#     # ----------------------------------
#     # Axis limits (fixed for stability)
#     # ----------------------------------

#     x_range = [time.min(), time.max()]
#     y_range = [feature.min(), feature.max()]

#     # ----------------------------------
#     # Initial figure
#     # ----------------------------------

#     fig = go.Figure()

#     # Full time series line
#     fig.add_trace(
#         go.Scatter(
#             x=time,
#             y=feature,
#             mode="lines+markers",
#             name="Feature"
#         )
#     )

#     # Moving point
#     fig.add_trace(
#         go.Scatter(
#             x=[time[0]],
#             y=[feature[0]],
#             mode="markers",
#             marker=dict(size=point_size),
#             name="Current time"
#         )
#     )

#     # ----------------------------------
#     # Frames
#     # ----------------------------------

#     frames = []

#     for t in range(T):

#         frames.append(
#             go.Frame(
#                 data=[

#                     go.Scatter(
#                         x=time,
#                         y=feature,
#                         mode="lines+markers"
#                     ),

#                     go.Scatter(
#                         x=[time[t]],
#                         y=[feature[t]],
#                         mode="markers",
#                         marker=dict(size=point_size)
#                     )

#                 ],
#                 name=str(t)
#             )
#         )

#     fig.frames = frames

#     # ----------------------------------
#     # Slider
#     # ----------------------------------

#     steps = []

#     for t in range(T):

#         steps.append(
#             dict(
#                 method="animate",
#                 args=[
#                     [str(t)],
#                     dict(
#                         mode="immediate",
#                         frame=dict(duration=frame_duration, redraw=True),
#                         transition=dict(duration=0)
#                     )
#                 ],
#                 label=str(t)
#             )
#         )

#     sliders = [
#         dict(
#             active=0,
#             pad={"t": 50},
#             steps=steps
#         )
#     ]

#     # ----------------------------------
#     # Layout
#     # ----------------------------------

#     fig.update_layout(

#         title=title,

#         xaxis=dict(title="Time", range=x_range),

#         yaxis=dict(title=y_label, range=y_range),
#         # yaxis=dict(title="Fractal Dimension", range=y_range),

#         sliders=sliders,

#         updatemenus=[
#             dict(
#                 type="buttons",
#                 showactive=False,

#                 buttons=[

#                     dict(
#                         label="Play",
#                         method="animate",
#                         args=[
#                             None,
#                             dict(
#                                 frame=dict(duration=frame_duration, redraw=True),
#                                 transition=dict(duration=0),
#                                 fromcurrent=True
#                             )
#                         ]
#                     ),

#                     dict(
#                         label="Pause",
#                         method="animate",
#                         args=[
#                             [None],
#                             dict(
#                                 frame=dict(duration=0),
#                                 transition=dict(duration=0),
#                                 mode="immediate"
#                             )
#                         ]
#                     )

#                 ]
#             )
#         ]
#     )

#     return fig



# ----------------------------
# 0. Preprocessing
# ----------------------------
def normalize_curve(r):
    r = np.asarray(r)
    r = r - r.mean(axis=0)
    r = r / (np.std(r) + 1e-8)
    return r


def match_length(r1, r2):
    """Trim to same length (simple version)"""
    T = min(len(r1), len(r2))
    return r1[:T], r2[:T]


# ----------------------------
# 1. Position Cross-Correlation
# ----------------------------
def position_cross_correlation(r1, r2, max_lag):
    r1, r2 = match_length(normalize_curve(r1), normalize_curve(r2))
    
    C = []
    for lag in range(max_lag):
        if lag == 0:
            val = np.mean(np.sum(r1 * r2, axis=1))
        else:
            val = np.mean(np.sum(r1[:-lag] * r2[lag:], axis=1))
        C.append(val)
    
    return np.array(C)


# ----------------------------
# 2. Velocity Cross-Correlation
# ----------------------------
def velocity_cross_correlation(r1, r2, max_lag):
    r1, r2 = match_length(normalize_curve(r1), normalize_curve(r2))
    
    v1 = np.diff(r1, axis=0)
    v2 = np.diff(r2, axis=0)
    
    C = []
    for lag in range(max_lag):
        if lag == 0:
            val = np.mean(np.sum(v1 * v2, axis=1))
        else:
            val = np.mean(np.sum(v1[:-lag] * v2[lag:], axis=1))
        C.append(val)
    
    C = np.array(C)
    return C / (C[0] + 1e-8)


# ----------------------------
# 3. Tangent Cross-Correlation
# ----------------------------
def tangent_cross_correlation(r1, r2, max_lag):
    r1, r2 = match_length(normalize_curve(r1), normalize_curve(r2))
    
    v1 = np.diff(r1, axis=0)
    v2 = np.diff(r2, axis=0)
    
    t1 = v1 / (np.linalg.norm(v1, axis=1, keepdims=True) + 1e-8)
    t2 = v2 / (np.linalg.norm(v2, axis=1, keepdims=True) + 1e-8)
    
    C = []
    for lag in range(max_lag):
        if lag == 0:
            val = np.mean(np.sum(t1 * t2, axis=1))
        else:
            val = np.mean(np.sum(t1[:-lag] * t2[lag:], axis=1))
        C.append(val)
    
    return np.array(C)


# ----------------------------
# 4. Displacement Cross-Correlation
# ----------------------------
def displacement_cross_correlation(r1, r2, max_lag):
    r1, r2 = match_length(normalize_curve(r1), normalize_curve(r2))
    
    D = []
    for lag in range(1, max_lag + 1):
        d1 = r1[lag:] - r1[:-lag]
        d2 = r2[lag:] - r2[:-lag]
        
        val = np.mean(np.sum(d1 * d2, axis=1))
        D.append(val)
    
    D = np.array(D)
    return D / (np.max(np.abs(D)) + 1e-8)


# ----------------------------
# 5. Spectral Cross-Correlation
# ----------------------------
def spectral_cross_correlation(r1, r2, n_freq):
    r1, r2 = match_length(normalize_curve(r1), normalize_curve(r2))
    
    S1 = np.abs(fft(r1, axis=0))**2
    S2 = np.abs(fft(r2, axis=0))**2
    
    S1 = S1.mean(axis=1)[:n_freq]
    S2 = S2.mean(axis=1)[:n_freq]
    
    # cosine similarity in frequency space
    num = np.dot(S1, S2)
    den = np.linalg.norm(S1) * np.linalg.norm(S2) + 1e-8
    
    return num / den


# ----------------------------
# 6. Unified Cross-Descriptor
# ----------------------------
def cross_descriptor(r1, r2, max_lag=50, n_freq=30):
    Cp = position_cross_correlation(r1, r2, max_lag)
    Cv = velocity_cross_correlation(r1, r2, max_lag)
    Ct = tangent_cross_correlation(r1, r2, max_lag)
    D  = displacement_cross_correlation(r1, r2, max_lag)
    S  = np.array([spectral_cross_correlation(r1, r2, n_freq)])
    
    return np.concatenate([Cp, Cv, Ct, D, S])


# ----------------------------
# Example usage
# ----------------------------
if __name__ == "__main__":
    T = 300
    t = np.linspace(0, 10, T)
    
    r1 = np.stack([np.sin(t), np.cos(t), np.sin(2*t)], axis=1)
    r2 = np.stack([np.sin(t+0.5), np.cos(t+0.5), np.sin(2*t+0.5)], axis=1)
    
    descriptor = cross_descriptor(r1, r2)
    
    print("Cross-descriptor shape:", descriptor.shape)

def features(curva):
    X=preprocess([curva.x,curva.y,curva.z], smooth=False)
    X=np.array(X)
    X=X.T
    V, A = kinematics(X, 1)
    speed = np.linalg.norm(V, axis=1)
    accel = np.linalg.norm(A, axis=1)
    J = jerk(A, 1)
    normJerk=np.linalg.norm(J)
    per=periodicity_measure(X)
    ent=signal_entropy(X)
    kappa=curvature(V,A)
    print('periodicidad: ',periodicity_measure(X))#,periodicity_measure(Y),periodicity_measure(Z))
    print('entropia: ',signal_entropy(X))#,signal_entropy(Y),signal_entropy(Z))
    print('Primera Derivada: ', speed.mean(),speed.std(),speed.max())
    print('Segunda Derivada: ', accel.mean(),accel.std(),accel.max())
    print('Tercera Derivada: ', normJerk.mean(),normJerk.std(),normJerk.max())
    print('curvatura: ', np.nanmean(kappa), np.nanstd(kappa), np.nanmax(kappa))

    # return speed,accel,J,per,ent,k

#################################



# import numpy as np
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots

# from scipy.signal import find_peaks
# from scipy.ndimage import gaussian_filter1d


# =========================================================
# CURVATURE
# =========================================================

# def compute_curvature(
#     df,
#     x_col="x",
#     y_col="y",
#     z_col="z",
#     smooth_sigma=1
# ):

#     x = df[x_col].values
#     y = df[y_col].values
#     z = df[z_col].values

#     if smooth_sigma is not None:

#         x = gaussian_filter1d(x, smooth_sigma)
#         y = gaussian_filter1d(y, smooth_sigma)
#         z = gaussian_filter1d(z, smooth_sigma)

#     r = np.stack([x, y, z], axis=1)

#     dr = np.gradient(r, axis=0)
#     d2r = np.gradient(dr, axis=0)

#     cross = np.cross(dr, d2r)

#     num = np.linalg.norm(cross, axis=1)
#     den = np.linalg.norm(dr, axis=1)**3 + 1e-8

#     return num / den


# =========================================================
# PCA PLANE FIT
# =========================================================

def fit_plane_pca(
    df,
    x_col="x",
    y_col="y",
    z_col="z"
):

    X = df[[x_col, y_col, z_col]].values

    centroid = X.mean(axis=0)

    Xc = X - centroid

    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)

    basis1 = Vt[0]
    basis2 = Vt[1]

    normal = Vt[2]

    return centroid, basis1, basis2, normal


# =========================================================
# PLANE MESH
# =========================================================

def generate_plane_mesh(
    centroid,
    basis1,
    basis2,
    scale=10,
    resolution=20
):

    u = np.linspace(-scale, scale, resolution)
    v = np.linspace(-scale, scale, resolution)

    U, V = np.meshgrid(u, v)

    plane = (
        centroid
        + U[..., None] * basis1
        + V[..., None] * basis2
    )

    Xp = plane[..., 0]
    Yp = plane[..., 1]
    Zp = plane[..., 2]

    return Xp, Yp, Zp


# =========================================================
# MAIN FUNCTION
# =========================================================

def animate_flagella_feature_curvature_plane(
    cell_data,
    feature,
    time=None,
    x_col="x",
    y_col="y",
    z_col="z",
    feature_name="Feature",
    frame_duration=80,
    smooth_sigma=1,
    peak_prominence=0.1,
    peak_distance=5,
    plane_opacity=0.35,
    plane_scale=None,
    head_marker_size=10,
    max_peak_buttons=12,
    fix_view=True,
    camera_eye=dict(x=1.8, y=1.8, z=1.2),

):
    """
    TOP:
        3D flagellum
        colored by curvature
        with animated PCA best-fit plane

    BOTTOM:
        Feature time series
        moving point
        peak markers

    Includes:
        - slider
        - play/pause
        - peak jump dropdown
        - stable PCA plane visualization
    """

    feature = np.asarray(feature)

    T = len(cell_data)

    if time is None:
        time = np.arange(T)

    time = np.asarray(time)

    # =====================================================
    # PEAK DETECTION
    # =====================================================

    peaks, _ = find_peaks(
        feature,
        prominence=peak_prominence,
        distance=peak_distance
    )

    # =====================================================
    # GLOBAL LIMITS
    # =====================================================

    all_x, all_y, all_z = [], [], []

    for df in cell_data:

        x = df[x_col].values
        y = df[y_col].values
        z = df[z_col].values

        all_x.extend(x)
        all_y.extend(y)
        all_z.extend(z)

    x_range = [min(all_x), max(all_x)]
    y_range = [min(all_y), max(all_y)]
    z_range = [min(all_z), max(all_z)]

    # =====================================================
    # AUTOMATIC PLANE SCALE
    # =====================================================

    if plane_scale is None:

        dx = x_range[1] - x_range[0]
        dy = y_range[1] - y_range[0]
        dz = z_range[1] - z_range[0]

        plane_scale = 0.35 * max(dx, dy, dz)

    # =====================================================
    # CURVATURE PRECOMPUTATION
    # =====================================================

    curvatures = []

    for df in cell_data:

        kappa = compute_curvature(
            df,
            x_col,
            y_col,
            z_col,
            smooth_sigma
        )

        curvatures.append(kappa)

    all_kappa = np.concatenate(curvatures)

    global_kmin = 0
    global_kmax = np.percentile(all_kappa, 99)

    # =====================================================
    # INITIAL FRAME
    # =====================================================

    df0 = cell_data[0].copy()

    if smooth_sigma is not None:

        df0[x_col] = gaussian_filter1d(
            df0[x_col],
            smooth_sigma
        )

        df0[y_col] = gaussian_filter1d(
            df0[y_col],
            smooth_sigma
        )

        df0[z_col] = gaussian_filter1d(
            df0[z_col],
            smooth_sigma
        )

    centroid, e1, e2, normal = fit_plane_pca(
        df0,
        x_col,
        y_col,
        z_col
    )

    Xp, Yp, Zp = generate_plane_mesh(
        centroid,
        e1,
        e2,
        scale=plane_scale
    )

    k0 = curvatures[0]

    # =====================================================
    # FIGURE
    # =====================================================

    fig = make_subplots(
        rows=2,
        cols=1,

        row_heights=[0.7, 0.3],

        vertical_spacing=0.08,

        specs=[
            [{"type": "scene"}],
            [{"type": "xy"}]
        ],

        subplot_titles=(
            "3D Flagellum + PCA Plane",
            feature_name
        )
    )

    # -----------------------------------------------------
    # Curvature-colored flagellum
    # -----------------------------------------------------

    fig.add_trace(

        go.Scatter3d(
            x=df0[x_col],
            y=df0[y_col],
            z=df0[z_col],

            mode="lines",

            line=dict(
                width=6,
                color=k0,
                colorscale="Viridis",
                cmin=global_kmin,
                cmax=global_kmax,

                colorbar=dict(
                    title="Curvature"
                )
            ),

            name="Flagellum"
        ),

        row=1,
        col=1
    )

    # -----------------------------------------------------
    # Head marker
    # -----------------------------------------------------

    fig.add_trace(

        go.Scatter3d(
            x=[df0[x_col].iloc[0]],
            y=[df0[y_col].iloc[0]],
            z=[df0[z_col].iloc[0]],

            mode="markers",

            marker=dict(
                size=head_marker_size
            ),

            name="Head"
        ),

        row=1,
        col=1
    )

    # -----------------------------------------------------
    # PCA plane
    # -----------------------------------------------------

    fig.add_trace(

        go.Surface(
            x=Xp,
            y=Yp,
            z=Zp,

            opacity=plane_opacity,

            showscale=False,

            name="PCA Plane"
        ),

        row=1,
        col=1
    )

    # -----------------------------------------------------
    # Feature line
    # -----------------------------------------------------

    fig.add_trace(

        go.Scatter(
            x=time,
            y=feature,

            mode="lines",

            name=feature_name
        ),

        row=2,
        col=1
    )

    # -----------------------------------------------------
    # Moving point
    # -----------------------------------------------------

    fig.add_trace(

        go.Scatter(
            x=[time[0]],
            y=[feature[0]],

            mode="markers",

            marker=dict(size=12),

            name="Current"
        ),

        row=2,
        col=1
    )

    # -----------------------------------------------------
    # Peaks
    # -----------------------------------------------------

    fig.add_trace(

        go.Scatter(
            x=time[peaks],
            y=feature[peaks],

            mode="markers",

            marker=dict(
                size=10,
                symbol="diamond"
            ),

            name="Peaks"
        ),

        row=2,
        col=1
    )

    # =====================================================
    # FRAMES
    # =====================================================

    frames = []

    for t in range(T):

        df = cell_data[t].copy()

        if smooth_sigma is not None:

            df[x_col] = gaussian_filter1d(
                df[x_col],
                smooth_sigma
            )

            df[y_col] = gaussian_filter1d(
                df[y_col],
                smooth_sigma
            )

            df[z_col] = gaussian_filter1d(
                df[z_col],
                smooth_sigma
            )

        # ---------------------------------------------
        # Curvature
        # ---------------------------------------------

        kappa = curvatures[t]

        # ---------------------------------------------
        # PCA plane
        # ---------------------------------------------

        centroid, e1, e2, normal = fit_plane_pca(
            df,
            x_col,
            y_col,
            z_col
        )

        Xp, Yp, Zp = generate_plane_mesh(
            centroid,
            e1,
            e2,
            scale=plane_scale
        )

        # ---------------------------------------------
        # Frame
        # ---------------------------------------------

        frames.append(

            go.Frame(
                data=[

                    # Curvature-colored flagellum
                    go.Scatter3d(
                        x=df[x_col],
                        y=df[y_col],
                        z=df[z_col],

                        mode="lines",

                        line=dict(
                            width=6,
                            color=kappa,
                            colorscale="Viridis",
                            cmin=global_kmin,
                            cmax=global_kmax
                        )
                    ),

                    # Head
                    go.Scatter3d(
                        x=[df[x_col].iloc[0]],
                        y=[df[y_col].iloc[0]],
                        z=[df[z_col].iloc[0]],

                        mode="markers",

                        marker=dict(
                            size=head_marker_size
                        )
                    ),

                    # PCA plane
                    go.Surface(
                        x=Xp,
                        y=Yp,
                        z=Zp,

                        opacity=plane_opacity,

                        showscale=False
                    ),

                    # Feature line
                    go.Scatter(
                        x=time,
                        y=feature,

                        mode="lines"
                    ),

                    # Moving point
                    go.Scatter(
                        x=[time[t]],
                        y=[feature[t]],

                        mode="markers",

                        marker=dict(size=12)
                    ),

                    # Peaks
                    go.Scatter(
                        x=time[peaks],
                        y=feature[peaks],

                        mode="markers",

                        marker=dict(
                            size=10,
                            symbol="diamond"
                        )
                    )

                ],

                name=str(t)
            )
        )

    fig.frames = frames

    # =====================================================
    # SLIDER
    # =====================================================

    steps = []

    for t in range(T):

        steps.append(

            dict(
                method="animate",

                args=[
                    [str(t)],

                    dict(
                        mode="immediate",

                        frame=dict(
                            duration=frame_duration,
                            redraw=True
                        ),

                        transition=dict(duration=0)
                    )
                ],

                label=str(t)
            )
        )

    sliders = [
        dict(
            active=0,
            pad={"t": 50},
            steps=steps
        )
    ]

    # =====================================================
    # PEAK BUTTONS
    # =====================================================

    peak_buttons = []

    for i, p in enumerate(peaks[:max_peak_buttons]):

        peak_buttons.append(

            dict(
                label=f"Peak {i}",

                method="animate",

                args=[
                    [str(p)],

                    dict(
                        mode="immediate",

                        frame=dict(
                            duration=0,
                            redraw=True
                        ),

                        transition=dict(duration=0)
                    )
                ]
            )
        )

    camera = dict(
        eye=camera_eye
    )
    # =====================================================
    # LAYOUT
    # =====================================================

    fig.update_layout(
        uirevision="fixed",

        height=900,

        scene=dict(

            xaxis=dict(range=x_range),
            yaxis=dict(range=y_range),
            zaxis=dict(range=z_range),

            aspectmode="data",

            camera=camera if fix_view else None
        ),

        xaxis=dict(
            title="Time"
        ),

        yaxis=dict(
            title=feature_name,
            range=[
                feature.min(),
                feature.max()
            ]
        ),

        sliders=sliders,

        updatemenus=[

            # MAIN CONTROLS
            dict(
                type="buttons",

                showactive=False,

                direction="left",

                x=0.1,
                y=1.15,

                buttons=[

                    dict(
                        label="⏮ Prev",

                        method="animate",

                        args=[
                            None,

                            dict(
                                mode="immediate",

                                frame=dict(duration=0),

                                transition=dict(duration=0),

                                direction="reverse"
                            )
                        ]
                    ),

                    dict(
                        label="Play",

                        method="animate",

                        args=[
                            None,

                            dict(
                                frame=dict(
                                    duration=frame_duration
                                ),

                                transition=dict(duration=0),

                                fromcurrent=True
                            )
                        ]
                    ),

                    dict(
                        label="Pause",

                        method="animate",

                        args=[
                            [None],

                            dict(
                                frame=dict(duration=0),

                                transition=dict(duration=0),

                                mode="immediate"
                            )
                        ]
                    ),

                    dict(
                        label="⏭ Next",

                        method="animate",

                        args=[
                            None,

                            dict(
                                mode="immediate",

                                frame=dict(duration=0),

                                transition=dict(duration=0)
                            )
                        ]
                    )

                ]
            ),

            # PEAK DROPDOWN
            dict(
                type="dropdown",

                x=0.6,
                y=1.15,

                buttons=peak_buttons,

                showactive=True
            )

        ]
    )

    return fig

def petrosian_fd(x):
    """
    Compute Petrosian Fractal Dimension (PFD)

    Parameters
    ----------
    x : array-like, shape (N,)

    Returns
    -------
    float
    """
    x = np.asarray(x, dtype=float)
    N = x.size

    # First derivative
    diff = np.diff(x)

    # Sign of derivative
    sign_diff = np.sign(diff)

    # Count sign changes
    N_delta = np.sum(sign_diff[1:] * sign_diff[:-1] < 0)

    return np.log(N) / (np.log(N) + np.log(N / (N + 0.4 * N_delta)))

def higuchi_fd_fast(x, kmax=10):
    x = np.asarray(x, dtype=float)
    N = len(x)

    Lk = np.zeros(kmax)

    for k in range(1, kmax + 1):
        Lm = []

        for m in range(k):
            idx = np.arange(m, N, k)
            if idx.size < 2:
                continue

            diffs = np.abs(x[idx[1:]] - x[idx[:-1]])
            L = diffs.sum()

            norm = (N - 1) / ((idx.size - 1) * k)
            Lm.append(L * norm)

        if Lm:
            Lk[k - 1] = np.mean(Lm)
        else:
            Lk[k - 1] = np.nan

    valid = (Lk > 0) & ~np.isnan(Lk)

    log_k = np.log(1 / np.arange(1, kmax + 1)[valid])
    log_Lk = np.log(Lk[valid])

    return np.polyfit(log_k, log_Lk, 1)[0]

def freqsComp(idx1,idx2):
    indiceDistro1=idx1
    indiceDistro2=idx2
    fouFD1 = np.fft.fft(dfCelsZenodoDimFrac.dimFracDistro.iloc[indiceDistro1])
    frecsFD1=np.fft.fftfreq(len(dfCelsZenodoDimFrac.dimFracDistro.iloc[indiceDistro1]),1.0/90.0)
    suplim1=int(np.ceil(len(fouFD1)/2))
    maxPow1=np.argmax(np.abs(fouFD1[1:suplim1]))
    # print(frecsFD1[maxPow1+1])
    fouFD2 = np.fft.fft(dfCelsZenodoDimFrac.dimFracDistro.iloc[indiceDistro2])
    frecsFD2=np.fft.fftfreq(len(dfCelsZenodoDimFrac.dimFracDistro.iloc[indiceDistro2]),1.0/90.0)
    suplim2=int(np.ceil(len(fouFD2)/2))
    maxPow2=np.argmax(np.abs(fouFD2[1:suplim2]))
    # print(frecsFD2[maxPow2+1])
    if( frecsFD2[maxPow2+1] > frecsFD1[maxPow1+1] ):
        return frecsFD2[maxPow2+1]/frecsFD1[maxPow1+1]
    else:
        return frecsFD1[maxPow1+1]/frecsFD2[maxPow2+1]

import numpy as np
from scipy.fft import fft


# ----------------------------
# 0. Preprocessing
# ----------------------------
def normalize_curve(r):
    r = np.asarray(r)
    r = r - r.mean(axis=0)
    r = r / (np.std(r) + 1e-8)
    return r


def match_length(r1, r2):
    """Trim to same length (simple version)"""
    T = min(len(r1), len(r2))
    return r1[:T], r2[:T]


# ----------------------------
# 1. Position Cross-Correlation
# ----------------------------
def position_cross_correlation(r1, r2, max_lag):
    r1, r2 = match_length(normalize_curve(r1), normalize_curve(r2))
    
    C = []
    for lag in range(max_lag):
        if lag == 0:
            val = np.mean(np.sum(r1 * r2, axis=1))
        else:
            val = np.mean(np.sum(r1[:-lag] * r2[lag:], axis=1))
        C.append(val)
    
    return np.array(C)


# ----------------------------
# 2. Velocity Cross-Correlation
# ----------------------------
def velocity_cross_correlation(r1, r2, max_lag):
    r1, r2 = match_length(normalize_curve(r1), normalize_curve(r2))
    
    v1 = np.diff(r1, axis=0)
    v2 = np.diff(r2, axis=0)
    
    C = []
    for lag in range(max_lag):
        if lag == 0:
            val = np.mean(np.sum(v1 * v2, axis=1))
        else:
            val = np.mean(np.sum(v1[:-lag] * v2[lag:], axis=1))
        C.append(val)
    
    C = np.array(C)
    return C / (C[0] + 1e-8)


# ----------------------------
# 3. Tangent Cross-Correlation
# ----------------------------
def tangent_cross_correlation(r1, r2, max_lag):
    r1, r2 = match_length(normalize_curve(r1), normalize_curve(r2))
    
    v1 = np.diff(r1, axis=0)
    v2 = np.diff(r2, axis=0)
    
    t1 = v1 / (np.linalg.norm(v1, axis=1, keepdims=True) + 1e-8)
    t2 = v2 / (np.linalg.norm(v2, axis=1, keepdims=True) + 1e-8)
    
    C = []
    for lag in range(max_lag):
        if lag == 0:
            val = np.mean(np.sum(t1 * t2, axis=1))
        else:
            val = np.mean(np.sum(t1[:-lag] * t2[lag:], axis=1))
        C.append(val)
    
    return np.array(C)


# ----------------------------
# 4. Displacement Cross-Correlation
# ----------------------------
def displacement_cross_correlation(r1, r2, max_lag):
    r1, r2 = match_length(normalize_curve(r1), normalize_curve(r2))
    
    D = []
    for lag in range(1, max_lag + 1):
        d1 = r1[lag:] - r1[:-lag]
        d2 = r2[lag:] - r2[:-lag]
        
        val = np.mean(np.sum(d1 * d2, axis=1))
        D.append(val)
    
    D = np.array(D)
    return D / (np.max(np.abs(D)) + 1e-8)


# ----------------------------
# 5. Spectral Cross-Correlation
# ----------------------------
def spectral_cross_correlation(r1, r2, n_freq):
    r1, r2 = match_length(normalize_curve(r1), normalize_curve(r2))
    
    S1 = np.abs(fft(r1, axis=0))**2
    S2 = np.abs(fft(r2, axis=0))**2
    
    S1 = S1.mean(axis=1)[:n_freq]
    S2 = S2.mean(axis=1)[:n_freq]
    
    # cosine similarity in frequency space
    num = np.dot(S1, S2)
    den = np.linalg.norm(S1) * np.linalg.norm(S2) + 1e-8
    
    return num / den


# ----------------------------
# 6. Unified Cross-Descriptor
# ----------------------------
def cross_descriptor(r1, r2, max_lag=50, n_freq=30):
    Cp = position_cross_correlation(r1, r2, max_lag)
    Cv = velocity_cross_correlation(r1, r2, max_lag)
    Ct = tangent_cross_correlation(r1, r2, max_lag)
    D  = displacement_cross_correlation(r1, r2, max_lag)
    S  = np.array([spectral_cross_correlation(r1, r2, n_freq)])
    
    return np.concatenate([Cp, Cv, Ct, D, S])


# ----------------------------
# Example usage
# ----------------------------
if __name__ == "__main__":
    T = 300
    t = np.linspace(0, 10, T)
    
    r1 = np.stack([np.sin(t), np.cos(t), np.sin(2*t)], axis=1)
    r2 = np.stack([np.sin(t+0.5), np.cos(t+0.5), np.sin(2*t+0.5)], axis=1)
    
    descriptor = cross_descriptor(r1, r2)
    
    print("Cross-descriptor shape:", descriptor.shape)

# Main

First, the files with the space-curves coordinates are read relating the name of the cell with the list of dataframes containing its coordinates.

In [ ]:
# Se leen las coordenadas tridimensionales de las células en Z e n o d o
# homedir=os.path.expanduser('~')
homedir=os.path.expanduser('~')
root_dir = homedir+"/lastestxZenodo/traces_micrometers/"
#root_dir = "/home/sidney/lastestxZenodo/trace_microns_smooth/"
curvasFlagelaresZenodo=[]
# Iterate over all folders and files
for foldername, subfolders, filenames in os.walk(root_dir):
    # print(f"Current folder: {foldername}")
    # if(not filenames):
    #     temp=foldername.split('/')
    #     fecha=temp[-1]
    #     print('------>',fecha)
    if(filenames):
        temp=foldername.split('/')
        celula=temp[-1]
        # print('------>',celula)
        fileX=os.path.join(foldername,'X.csv')
        fileY=os.path.join(foldername,'Y.csv')
        fileZ=os.path.join(foldername,'Z.csv')
        xdf = pd.read_csv(fileX, sep=' ', header=None)
        ydf = pd.read_csv(fileY, sep=' ', header=None)
        zdf = pd.read_csv(fileZ, sep=' ', header=None)
        ntiempos=xdf.shape[1]
        tiempos=[]
        for i in range(ntiempos):
            tempdf=pd.concat([xdf[i],ydf[i],zdf[i]],axis=1)
            tempdf.columns=['x','y','z']
            cleanTempdf=tempdf.dropna()
            tiempos.append(cleanTempdf)
        curvasFlagelaresZenodo.append([celula,tiempos])

The following variable contains the space-curves coordinates without the names of the cells.

In [ ]:
flagellar_data = [item[1] for item in curvasFlagelaresZenodo]   # list of list of DataFrames, without the names

In [ ]:
ncels=len(curvasFlagelaresZenodo)
for i in range(ncels):
    if('210702' in curvasFlagelaresZenodo[i][0]):
        print(curvasFlagelaresZenodo[i][0])
    # if('210730' in curvasFlagelaresZenodo[i][0]):
    #     print(curvasFlagelaresZenodo[i][0])
    # if('181026' in curvasFlagelaresZenodo[i][0]):
    #     print(curvasFlagelaresZenodo[i][0])
    # if('181030' in curvasFlagelaresZenodo[i][0]):
    #     print(curvasFlagelaresZenodo[i][0])


In [ ]:
ncels=len(curvasFlagelaresZenodo)
for i in range(ncels):
    # if('210702' in curvasFlagelaresZenodo[i][0]):
    #     print(curvasFlagelaresZenodo[i][0])
    if('210730' in curvasFlagelaresZenodo[i][0]):
        print(curvasFlagelaresZenodo[i][0])
    # if('181026' in curvasFlagelaresZenodo[i][0]):
    #     print(curvasFlagelaresZenodo[i][0])
    # if('181030' in curvasFlagelaresZenodo[i][0]):
    #     print(curvasFlagelaresZenodo[i][0])


In [ ]:
ncels=len(curvasFlagelaresZenodo)
for i in range(ncels):
    # if('210702' in curvasFlagelaresZenodo[i][0]):
    #     print(curvasFlagelaresZenodo[i][0])
    # if('210730' in curvasFlagelaresZenodo[i][0]):
    #     print(curvasFlagelaresZenodo[i][0])
    if('181026' in curvasFlagelaresZenodo[i][0]):
        print(curvasFlagelaresZenodo[i][0])
    # if('181030' in curvasFlagelaresZenodo[i][0]):
    #     print(curvasFlagelaresZenodo[i][0])


In [ ]:
ncels=len(curvasFlagelaresZenodo)
for i in range(ncels):
    # if('210702' in curvasFlagelaresZenodo[i][0]):
    #     print(curvasFlagelaresZenodo[i][0])
    # if('210730' in curvasFlagelaresZenodo[i][0]):
    #     print(curvasFlagelaresZenodo[i][0])
    # if('181026' in curvasFlagelaresZenodo[i][0]):
    #     print(curvasFlagelaresZenodo[i][0])
    if('181030' in curvasFlagelaresZenodo[i][0]):
        print(curvasFlagelaresZenodo[i][0])


In [ ]:
ncap=0
nNoCap=0
for i in range(len(curvasFlagelaresZenodo)):
    if( 'No' in curvasFlagelaresZenodo[i][0] ):
        nNoCap+=1
    else:
        ncap+=1

print('|Cap|= ', ncap, ' which corresponds to ',ncap*100/len(curvasFlagelaresZenodo),'%')
print('|NoCap|= ', nNoCap, ' which corresponds to ',nNoCap*100/len(curvasFlagelaresZenodo),'%')
print(len(curvasFlagelaresZenodo))

In [ ]:
desplazamiento=[]
for i in range(len(curvasFlagelaresZenodo)):
    if( len(curvasFlagelaresZenodo[i][1]) >= 90 ):
        nom=curvasFlagelaresZenodo[i][0]
        dist=dist3D(curvasFlagelaresZenodo[i][1][0].iloc[0].x,curvasFlagelaresZenodo[i][1][0].iloc[0].y,curvasFlagelaresZenodo[i][1][0].iloc[0].z,
                    curvasFlagelaresZenodo[i][1][89].iloc[0].x,curvasFlagelaresZenodo[i][1][89].iloc[0].y,curvasFlagelaresZenodo[i][1][89].iloc[0].z)
        if( 'No' in curvasFlagelaresZenodo[i][0] ):
            cond='NoCap'
        else:
            cond='Cap'
        desplazamiento.append([nom,cond,dist])

In [ ]:
len(desplazamiento)

In [ ]:
dfDesplazamiento=pd.DataFrame(desplazamiento, columns=['cel','Condition','Displacement'])

In [ ]:
fig=px.violin(dfDesplazamiento, x='Displacement', color='Condition', box=True, points='all', #log_x=True,
             category_orders={"Condition":["NoCap","Cap"]})
fig.update_traces(meanline_visible=True)
fig.update_layout(title_text="Distribution of the Displacement of each Cell", 
# fig.update_layout(title_text='Distribución de la frecuencia principal para la serie de tiempo de cada célula', 
                 title_x=0.5,font=dict(size=20),xaxis_title="Displacement")
fig.show()

In [ ]:
dfDesplazamiento[dfDesplazamiento['Displacement'] > 46.86 ].sort_values(by=['Displacement'], ascending=False)

In [ ]:
dfDesplazamiento[dfDesplazamiento['Displacement'] < 15 ].sort_values(by=['Displacement'], ascending=False)

# Flagellar Fractal Dimension

<!--
Esta dimensión fractal flagelar es diferente al "flagellar curvature ratio" definido por Suarez en: Movement characteristics and acrosomal status of rabbit spermatozoa recovered at the site and time of fertilization Suarez 1983. Suarez lo define como *la distancia en línea recta desde la unión de la cabeza con la pieza media al primer punto de inflección de la cola, dividido por la distancia curvilinea entre esos mismo dos puntos*. 

El objetivo principal es al análisis de la **dimensión fractal flagelar D**, 
-->

This flagellar fractal dimension is different from the "flagellar curvature ratio" defined by Suarez in: Movement characteristics and acrosomal status of rabbit spermatozoa recovered at the site and time of fertilization, Suarez 1983. Suarez defines it as *the rectilinear distance from the head-middle pice junction to the first inflection point of the tail divided by the curvilinear distance among those two points*.

---

The main goal is the analysis of the Katz **space-curve's fractal dimension D**:

$$
D=\frac{\log(n)}{\log(n)+\log(d/L)}.\nonumber
$$

---

In [ ]:
len(curvasFlagelaresZenodo),len(curvasFlagelaresZenodo[0][1]),curvasFlagelaresZenodo[0][1][0].shape[0]

First, we compute the **Katz Fractal Dimension** <u>for each flagellar curve of every sperm cell</u>.

In [ ]:
''' Computation
# Computation of the Katz fractal dimension for each space curve in the dataset

ncels=len(curvasFlagelaresZenodo)
celsZenodoDimFrac=[]
for celula in range(ncels):
    ntiempos=len(curvasFlagelaresZenodo[celula][1])
    fracDimDistro=[]
    for tiempo in range(ntiempos):
        nptos=curvasFlagelaresZenodo[celula][1][tiempo].shape[0]   #400
# The computation is only for those space-curves with at least 342 points
        if( nptos >= 342 ):
            nptos=342
            lcl=curvDist3D(curvasFlagelaresZenodo[celula][1][tiempo],nptos)
            d=conjuntoDiametro3D(curvasFlagelaresZenodo[celula][1][tiempo],nptos)
            flglrDimFractal=np.log10(nptos-1)/(np.log10(nptos-1)+np.log10(d/lcl))
            fracDimDistro.append(flglrDimFractal)
    celname=curvasFlagelaresZenodo[celula][0]   #.nom.iloc[0].split('_stac')[0]
    celsZenodoDimFrac.append([celname,fracDimDistro])

# dfCelsZenodoDimFrac=pd.DataFrame(celsZenodoDimFrac,columns=['cel','dimFracDistro'])

with open('celsZenodoDimFrac.json', 'w') as f:
    json.dump(celsZenodoDimFrac, f)
    '''

In [ ]:
# the fractal dimensions previously computed are readed from a json file
with open(homedir+"/Research/3DCurves/data/celsZenodoDimFrac.json") as f:
    lst = json.load(f)

dfCelsZenodoDimFrac=pd.DataFrame(lst,columns=['cel','dimFracDistro'])

In [ ]:
dfCelsZenodoDimFrac.head()

Now, the next step is to explore wich kind of **analysis** could get the best statistical classfication. Of course, if the <u>nature of the data implies certain requirements</u> these should be considered over the purely statistical point of view.

# Mean fKFD

In general, it could be considered that the dynamic mean is the best way to describe the behavior of several individual elements. As such, the first attempt is to use the mean flagellar Katz Fractal Dimension to analyze the motion of the space curves.

# Normalized Maximum of Power Spectrum for the fKFD

In [ ]:
xt=range(0,len(dfCelsZenodoDimFrac.dimFracDistro.iloc[10]))
fig = px.line(x=np.array(xt)/90,y=dfCelsZenodoDimFrac.dimFracDistro.iloc[10], markers=True)
fig.update_layout(title_text='Fractal dimension time series of a single cell', 
                  title_x=0.5,font=dict(size=15))
fig.update_layout(
    xaxis_title="time",
    yaxis_title="KFD"
)
fig.show()

fouFD = np.fft.fft(dfCelsZenodoDimFrac.dimFracDistro.iloc[10])
frecsFD=np.fft.fftfreq(len(dfCelsZenodoDimFrac.dimFracDistro.iloc[10]),1.0/90.0)
suplim=int(np.ceil(len(fouFD)/2))
fig = px.line(x=frecsFD[1:suplim],y=np.abs(fouFD[1:suplim]**2), markers=True)
fig.update_layout(title_text='Power spectrum of the Fractal Dimension Time-Series of a single cell', 
                  title_x=0.5,font=dict(size=15))
fig.update_layout(
    xaxis_title="Frequency",
    yaxis_title="Power"
)
fig.show()

In [ ]:
ncels=dfCelsZenodoDimFrac.shape[0]
# ncels=dfCelsZenodoDimFrac.shape[0]
FDMaxAmpDistNor=[]
for i in range(ncels):
    nom=dfCelsZenodoDimFrac.cel.iloc[i]
    fouFD = np.fft.fft(dfCelsZenodoDimFrac.dimFracDistro.iloc[i])
    frequency_amplitudes = np.abs(fouFD**2)
    suma=np.sum(frequency_amplitudes[1:])
    maxPow=np.max(frequency_amplitudes[1:])
    freqContribution=maxPow/suma
    # fdMaxMFw=np.max(dfCelsZenodoDimFrac.dimFracDistro.iloc[i])*freqContribution#(np.std(dfCelsZenodoDimFrac.dimFracDistro.iloc[i])*freqContribution)
    if( 'No' in dfCelsZenodoDimFrac.cel.iloc[i] ):
        cond='NoCap'
    else:
        cond='Cap'
    FDMaxAmpDistNor.append([nom,cond,freqContribution])

dfZenodoFDMaxAmpDistNor=pd.DataFrame(FDMaxAmpDistNor, columns=['cel','Condition','maxAmp'])

In [ ]:
fig=px.violin(dfZenodoFDMaxAmpDistNor, x='maxAmp', color='Condition', box=True, points='all', #log_x=True,
             category_orders={"Condition":["NoCap","Cap"]})
fig.update_traces(meanline_visible=True)
fig.update_layout(title_text="Distribution of the Normalized Maximum in the Power Spectrum of each Cell", 
# fig.update_layout(title_text='Distribución de la frecuencia principal para la serie de tiempo de cada célula', 
                 title_x=0.5,font=dict(size=20),xaxis_title="Normalized Max of Power Spectrum")
fig.show()

In [ ]:
dfZenodoFDMaxAmpDistNor[dfZenodoFDMaxAmpDistNor['maxAmp']> 0.226 ].sort_values(by=['maxAmp'], ascending=False)

In [ ]:
dfZenodoFDMaxAmpDistNor[dfZenodoFDMaxAmpDistNor['maxAmp'] < 0.05 ].sort_values(by=['maxAmp'], ascending=False)

In [ ]:
## Sperm-7-NoCap_210702_Exp15_cell-1

indiceDistro=16
for i in range(len(curvasFlagelaresZenodo)):
    if( curvasFlagelaresZenodo[i][0] == dfCelsZenodoDimFrac.cel.iloc[indiceDistro] ):
        print(i,indiceDistro)
        indiceFlagelar=i
print(dfCelsZenodoDimFrac.cel.iloc[indiceDistro],curvasFlagelaresZenodo[indiceFlagelar][0])
feature=dfCelsZenodoDimFrac.dimFracDistro.iloc[indiceDistro]

cell_data=flagellar_data[indiceFlagelar]
dt = 1/90

time = np.arange(len(feature)) * dt


fig = animate_flagella_feature_curvature_plane(
    cell_data[:len(feature)],
    feature,
    frame_duration=300,
    plane_opacity=0.4,
    smooth_sigma=1,
    peak_prominence=0.02,
    peak_distance=5,
    camera_eye=dict(x=-2, y=1.8, z=2)
)

fig.show()


In [ ]:
# indiceDistro=16
# for i in range(len(curvasFlagelaresZenodo)):
#     if( curvasFlagelaresZenodo[i][0] == dfCelsZenodoDimFrac.cel.iloc[indiceDistro] ):
#         print(i,indiceDistro)
#         indiceFlagelar=i
# cell_data=flagellar_data[indiceFlagelar]

# normals = []

# for df in cell_data:

#     centroid, e1, e2, normal = fit_plane_pca(df)

#     normal = normal / np.linalg.norm(normal)

#     normals.append(normal)

# normals = np.array(normals)
# for i in range(1, len(normals)):

#     if np.dot(normals[i-1], normals[i]) < 0:
#         normals[i] *= -1
# n0 = normals[0]

# plane_angle = np.degrees(
#     np.arccos(
#         np.clip(normals @ n0, -1, 1)
#     )
# )
# nx, ny, nz = normals.T

# plane_azimuth = np.degrees(
#     np.arctan2(ny, nx)
# )

# plane_elevation = np.degrees(
#     np.arcsin(nz)
# )

In [ ]:
lim=100
indiceDistro=16
for i in range(len(curvasFlagelaresZenodo)):
    if( curvasFlagelaresZenodo[i][0] == dfCelsZenodoDimFrac.cel.iloc[indiceDistro] ):
        print(i,indiceDistro)
        indiceFlagelar=i
cell_data=flagellar_data[indiceFlagelar]

normals = []

for df in cell_data:

    centroid, e1, e2, normal = fit_plane_pca(df)

    normal = normal / np.linalg.norm(normal)

    normals.append(normal)

normals = np.array(normals)

for i in range(1, len(normals)):

    if np.dot(normals[i-1], normals[i]) < 0:
        normals[i] *= -1

nx, ny, nz = normals.T
fig=go.Figure()
fig.update_layout(
    autosize=False,
    width=1250,
    height=1000,
    scene_aspectmode='cube'
    # margin=dict(l=10, r=10, t=10, b=10, pad=10)
)
fig.add_trace(go.Scatter3d(
    x=nx[:lim],
    y=ny[:lim],
    z=nz[:lim],
    mode="lines+markers"
))
fig.add_trace(
    go.Scatter3d(
    x=[nx[0],nx[1]],
    y=[ny[0],ny[1]],
    z=[nz[0],nz[1]],
    mode='markers+lines',marker=dict(color='red')
)
)
fig.show()

n0 = normals[0]

plane_angle = np.degrees(
    np.arccos(
        np.clip(normals @ n0, -1, 1)
    )
)

fig = px.line(y=plane_angle, markers=True)
# fig.update_layout(title_text='Power spectrum of the Fractal Dimension Time-Series for '+
#                   dfCelsZenodoDimFrac.cel.iloc[indiceDistro], 
#                   title_x=0.5,font=dict(size=15))
# fig.update_layout(
#     xaxis_title="Frequency",
#     yaxis_title="Power"
# )
fig.show()
print(petrosian_fd(plane_angle))
print(higuchi_fd_fast(plane_angle))
print(periodicity_measure(plane_angle))

fouFD = np.fft.fft(plane_angle)
frecsFD=np.fft.fftfreq(len(plane_angle),1.0/90.0)
suplim=int(np.ceil(len(fouFD)/2))
fig = px.line(x=frecsFD[1:suplim],y=np.abs(fouFD[1:suplim]**2), markers=True)
fig.update_layout(title_text='Power spectrum of the Fractal Dimension Time-Series for ', 
                  title_x=0.5,font=dict(size=15))
fig.update_layout(
    xaxis_title="Frequency",
    yaxis_title="Power"
)
fig.show()

fouFD = np.fft.fft(plane_angle)
frequency_amplitudes = np.abs(fouFD**2)
suma=np.sum(frequency_amplitudes[1:])
maxPow=np.max(frequency_amplitudes[1:])
freqContribution=maxPow/suma
print(freqContribution)

## Sperm-1-Cap_170607_Exp2

In [ ]:
indiceDistro=58
for i in range(len(curvasFlagelaresZenodo)):
    if( curvasFlagelaresZenodo[i][0] == dfCelsZenodoDimFrac.cel.iloc[indiceDistro] ):
        print(i,indiceDistro)
        indiceFlagelar=i
print(dfCelsZenodoDimFrac.cel.iloc[indiceDistro],curvasFlagelaresZenodo[indiceFlagelar][0])
feature=dfCelsZenodoDimFrac.dimFracDistro.iloc[indiceDistro]

cell_data=flagellar_data[indiceFlagelar]
dt = 1/90

time = np.arange(len(feature)) * dt


fig = animate_flagella_feature_curvature_plane(
    cell_data[:len(feature)],
    feature,
    frame_duration=300,
    plane_opacity=0.4,
    smooth_sigma=1,
    peak_prominence=0.02,
    peak_distance=5,
    camera_eye=dict(x=-2, y=1.8, z=2)
)

fig.show()


In [ ]:
lim=100
indiceDistro=58
for i in range(len(curvasFlagelaresZenodo)):
    if( curvasFlagelaresZenodo[i][0] == dfCelsZenodoDimFrac.cel.iloc[indiceDistro] ):
        print(i,indiceDistro)
        indiceFlagelar=i
cell_data=flagellar_data[indiceFlagelar]

normals = []

for df in cell_data:

    centroid, e1, e2, normal = fit_plane_pca(df)

    normal = normal / np.linalg.norm(normal)

    normals.append(normal)

normals = np.array(normals)

for i in range(1, len(normals)):

    if np.dot(normals[i-1], normals[i]) < 0:
        normals[i] *= -1

nx, ny, nz = normals.T
fig=go.Figure()
fig.update_layout(
    autosize=False,
    width=1250,
    height=1000,
    scene_aspectmode='cube'
    # margin=dict(l=10, r=10, t=10, b=10, pad=10)
)
fig.add_trace(go.Scatter3d(
    x=nx[:lim],
    y=ny[:lim],
    z=nz[:lim],
    mode="lines+markers"
))
fig.add_trace(
    go.Scatter3d(
    x=[nx[0],nx[1]],
    y=[ny[0],ny[1]],
    z=[nz[0],nz[1]],
    mode='markers+lines',marker=dict(color='red')
)
)
fig.show()

n0 = normals[0]

plane_angle = np.degrees(
    np.arccos(
        np.clip(normals @ n0, -1, 1)
    )
)

fig = px.line(y=plane_angle, markers=True)
# fig.update_layout(title_text='Power spectrum of the Fractal Dimension Time-Series for '+
#                   dfCelsZenodoDimFrac.cel.iloc[indiceDistro], 
#                   title_x=0.5,font=dict(size=15))
# fig.update_layout(
#     xaxis_title="Frequency",
#     yaxis_title="Power"
# )
fig.show()

print(petrosian_fd(plane_angle))
print(higuchi_fd_fast(plane_angle))
print(periodicity_measure(plane_angle))

fouFD = np.fft.fft(plane_angle)
frecsFD=np.fft.fftfreq(len(plane_angle),1.0/90.0)
suplim=int(np.ceil(len(fouFD)/2))
fig = px.line(x=frecsFD[1:suplim],y=np.abs(fouFD[1:suplim]**2), markers=True)
fig.update_layout(title_text='Power spectrum of the Fractal Dimension Time-Series for ', 
                  title_x=0.5,font=dict(size=15))
fig.update_layout(
    xaxis_title="Frequency",
    yaxis_title="Power"
)
fig.show()

fouFD = np.fft.fft(plane_angle)
frequency_amplitudes = np.abs(fouFD**2)
suma=np.sum(frequency_amplitudes[1:])
maxPow=np.max(frequency_amplitudes[1:])
freqContribution=maxPow/suma
print(freqContribution)

## Sperm-11-NoCap_210702_Exp25

In [ ]:
indiceDistro=29
for i in range(len(curvasFlagelaresZenodo)):
    if( curvasFlagelaresZenodo[i][0] == dfCelsZenodoDimFrac.cel.iloc[indiceDistro] ):
        print(i,indiceDistro)
        indiceFlagelar=i
print(dfCelsZenodoDimFrac.cel.iloc[indiceDistro],curvasFlagelaresZenodo[indiceFlagelar][0])
feature=dfCelsZenodoDimFrac.dimFracDistro.iloc[indiceDistro]

cell_data=flagellar_data[indiceFlagelar]
dt = 1/90

time = np.arange(len(feature)) * dt


fig = animate_flagella_feature_curvature_plane(
    cell_data[:len(feature)],
    feature,
    frame_duration=300,
    plane_opacity=0.4,
    smooth_sigma=1,
    peak_prominence=0.02,
    peak_distance=5,
    camera_eye=dict(x=-2, y=1.8, z=2)
)

fig.show()


In [ ]:
lim=100
indiceDistro=29
for i in range(len(curvasFlagelaresZenodo)):
    if( curvasFlagelaresZenodo[i][0] == dfCelsZenodoDimFrac.cel.iloc[indiceDistro] ):
        print(i,indiceDistro)
        indiceFlagelar=i
cell_data=flagellar_data[indiceFlagelar]

normals = []

for df in cell_data:

    centroid, e1, e2, normal = fit_plane_pca(df)

    normal = normal / np.linalg.norm(normal)

    normals.append(normal)

normals = np.array(normals)

for i in range(1, len(normals)):

    if np.dot(normals[i-1], normals[i]) < 0:
        normals[i] *= -1

nx, ny, nz = normals.T
fig=go.Figure()
fig.update_layout(
    autosize=False,
    width=1250,
    height=1000,
    scene_aspectmode='cube'
    # margin=dict(l=10, r=10, t=10, b=10, pad=10)
)
fig.add_trace(go.Scatter3d(
    x=nx[:lim],
    y=ny[:lim],
    z=nz[:lim],
    mode="lines+markers"
))
fig.add_trace(
    go.Scatter3d(
    x=[nx[0],nx[1]],
    y=[ny[0],ny[1]],
    z=[nz[0],nz[1]],
    mode='markers+lines',marker=dict(color='red')
)
)
fig.show()

n0 = normals[0]

plane_angle = np.degrees(
    np.arccos(
        np.clip(normals @ n0, -1, 1)
    )
)

fig = px.line(y=plane_angle, markers=True)
# fig.update_layout(title_text='Power spectrum of the Fractal Dimension Time-Series for '+
#                   dfCelsZenodoDimFrac.cel.iloc[indiceDistro], 
#                   title_x=0.5,font=dict(size=15))
# fig.update_layout(
#     xaxis_title="Frequency",
#     yaxis_title="Power"
# )
fig.show()

print(petrosian_fd(plane_angle))
print(higuchi_fd_fast(plane_angle))
print(periodicity_measure(plane_angle))

fouFD = np.fft.fft(plane_angle)
frecsFD=np.fft.fftfreq(len(plane_angle),1.0/90.0)
suplim=int(np.ceil(len(fouFD)/2))
fig = px.line(x=frecsFD[1:suplim],y=np.abs(fouFD[1:suplim]**2), markers=True)
fig.update_layout(title_text='Power spectrum of the Fractal Dimension Time-Series for ', 
                  title_x=0.5,font=dict(size=15))
fig.update_layout(
    xaxis_title="Frequency",
    yaxis_title="Power"
)
fig.show()

fouFD = np.fft.fft(plane_angle)
frequency_amplitudes = np.abs(fouFD**2)
suma=np.sum(frequency_amplitudes[1:])
maxPow=np.max(frequency_amplitudes[1:])
freqContribution=maxPow/suma
print(freqContribution)

## Sperm-9-Cap_181030_Exp25

In [ ]:
indiceDistro=108
for i in range(len(curvasFlagelaresZenodo)):
    if( curvasFlagelaresZenodo[i][0] == dfCelsZenodoDimFrac.cel.iloc[indiceDistro] ):
        print(i,indiceDistro)
        indiceFlagelar=i
print(dfCelsZenodoDimFrac.cel.iloc[indiceDistro],curvasFlagelaresZenodo[indiceFlagelar][0])
feature=dfCelsZenodoDimFrac.dimFracDistro.iloc[indiceDistro]

cell_data=flagellar_data[indiceFlagelar]
dt = 1/90

time = np.arange(len(feature)) * dt


fig = animate_flagella_feature_curvature_plane(
    cell_data,
    feature,
    frame_duration=300,
    plane_opacity=0.4,
    smooth_sigma=1,
    peak_prominence=0.02,
    peak_distance=5,
    camera_eye=dict(x=-2, y=1.8, z=2)
)

fig.show()


In [ ]:
lim=100
indiceDistro=108
for i in range(len(curvasFlagelaresZenodo)):
    if( curvasFlagelaresZenodo[i][0] == dfCelsZenodoDimFrac.cel.iloc[indiceDistro] ):
        print(i,indiceDistro)
        indiceFlagelar=i
cell_data=flagellar_data[indiceFlagelar]

normals = []

for df in cell_data:

    centroid, e1, e2, normal = fit_plane_pca(df)

    normal = normal / np.linalg.norm(normal)

    normals.append(normal)

normals = np.array(normals)

for i in range(1, len(normals)):

    if np.dot(normals[i-1], normals[i]) < 0:
        normals[i] *= -1

nx, ny, nz = normals.T
fig=go.Figure()
fig.update_layout(
    autosize=False,
    width=1250,
    height=1000,
    scene_aspectmode='cube'
    # margin=dict(l=10, r=10, t=10, b=10, pad=10)
)
fig.add_trace(go.Scatter3d(
    x=nx[:lim],
    y=ny[:lim],
    z=nz[:lim],
    mode="lines+markers"
))
fig.add_trace(
    go.Scatter3d(
    x=[nx[0],nx[1]],
    y=[ny[0],ny[1]],
    z=[nz[0],nz[1]],
    mode='markers+lines',marker=dict(color='red')
)
)
fig.show()

n0 = normals[0]

plane_angle = np.degrees(
    np.arccos(
        np.clip(normals @ n0, -1, 1)
    )
)

fig = px.line(y=plane_angle, markers=True)
# fig.update_layout(title_text='Power spectrum of the Fractal Dimension Time-Series for '+
#                   dfCelsZenodoDimFrac.cel.iloc[indiceDistro], 
#                   title_x=0.5,font=dict(size=15))
# fig.update_layout(
#     xaxis_title="Frequency",
#     yaxis_title="Power"
# )
fig.show()
print(petrosian_fd(plane_angle))
print(higuchi_fd_fast(plane_angle))
print(periodicity_measure(plane_angle))

fouFD = np.fft.fft(plane_angle)
frecsFD=np.fft.fftfreq(len(plane_angle),1.0/90.0)
suplim=int(np.ceil(len(fouFD)/2))
fig = px.line(x=frecsFD[1:suplim],y=np.abs(fouFD[1:suplim]**2), markers=True)
fig.update_layout(title_text='Power spectrum of the Fractal Dimension Time-Series for ', 
                  title_x=0.5,font=dict(size=15))
fig.update_layout(
    xaxis_title="Frequency",
    yaxis_title="Power"
)
fig.show()

fouFD = np.fft.fft(plane_angle)
frequency_amplitudes = np.abs(fouFD**2)
suma=np.sum(frequency_amplitudes[1:])
maxPow=np.max(frequency_amplitudes[1:])
freqContribution=maxPow/suma
print(freqContribution)

## Sperm-4-Cap_190326_Exp7_cell-2

In [ ]:
indiceDistro=62
for i in range(len(curvasFlagelaresZenodo)):
    if( curvasFlagelaresZenodo[i][0] == dfCelsZenodoDimFrac.cel.iloc[indiceDistro] ):
        print(i,indiceDistro)
        indiceFlagelar=i
print(dfCelsZenodoDimFrac.cel.iloc[indiceDistro],curvasFlagelaresZenodo[indiceFlagelar][0])
feature=dfCelsZenodoDimFrac.dimFracDistro.iloc[indiceDistro]

cell_data=flagellar_data[indiceFlagelar]
dt = 1/90

time = np.arange(len(feature)) * dt


fig = animate_flagella_feature_curvature_plane(
    cell_data,
    feature,
    frame_duration=300,
    plane_opacity=0.4,
    smooth_sigma=1,
    peak_prominence=0.02,
    peak_distance=5,
    camera_eye=dict(x=-2, y=1.8, z=2)
)

fig.show()


In [ ]:
lim=200
indiceDistro=62
for i in range(len(curvasFlagelaresZenodo)):
    if( curvasFlagelaresZenodo[i][0] == dfCelsZenodoDimFrac.cel.iloc[indiceDistro] ):
        print(i,indiceDistro)
        indiceFlagelar=i
cell_data=flagellar_data[indiceFlagelar]

normals = []

for df in cell_data:

    centroid, e1, e2, normal = fit_plane_pca(df)

    normal = normal / np.linalg.norm(normal)

    normals.append(normal)

normals = np.array(normals)

for i in range(1, len(normals)):

    if np.dot(normals[i-1], normals[i]) < 0:
        normals[i] *= -1

nx, ny, nz = normals.T
fig=go.Figure()
fig.update_layout(
    autosize=False,
    width=1250,
    height=1000,
    scene_aspectmode='cube'
    # margin=dict(l=10, r=10, t=10, b=10, pad=10)
)
fig.add_trace(go.Scatter3d(
    x=nx[:lim],
    y=ny[:lim],
    z=nz[:lim],
    mode="lines+markers"
))
fig.add_trace(
    go.Scatter3d(
    x=[nx[0],nx[1]],
    y=[ny[0],ny[1]],
    z=[nz[0],nz[1]],
    mode='markers+lines',marker=dict(color='red')
)
)
fig.show()

n0 = normals[0]

plane_angle = np.degrees(
    np.arccos(
        np.clip(normals @ n0, -1, 1)
    )
)

fig = px.line(y=plane_angle, markers=True)
# fig.update_layout(title_text='Power spectrum of the Fractal Dimension Time-Series for '+
#                   dfCelsZenodoDimFrac.cel.iloc[indiceDistro], 
#                   title_x=0.5,font=dict(size=15))
# fig.update_layout(
#     xaxis_title="Frequency",
#     yaxis_title="Power"
# )
fig.show()
print(petrosian_fd(plane_angle))
print(higuchi_fd_fast(plane_angle))
print(periodicity_measure(plane_angle))

fouFD = np.fft.fft(plane_angle)
frecsFD=np.fft.fftfreq(len(plane_angle),1.0/90.0)
suplim=int(np.ceil(len(fouFD)/2))
fig = px.line(x=frecsFD[1:suplim],y=np.abs(fouFD[1:suplim]**2), markers=True)
fig.update_layout(title_text='Power spectrum of the Fractal Dimension Time-Series for ', 
                  title_x=0.5,font=dict(size=15))
fig.update_layout(
    xaxis_title="Frequency",
    yaxis_title="Power"
)
fig.show()

fouFD = np.fft.fft(plane_angle)
frequency_amplitudes = np.abs(fouFD**2)
suma=np.sum(frequency_amplitudes[1:])
maxPow=np.max(frequency_amplitudes[1:])
freqContribution=maxPow/suma
print(freqContribution)

## Sperm-8-Cap_190326_Exp12

In [ ]:
indiceDistro=61
for i in range(len(curvasFlagelaresZenodo)):
    if( curvasFlagelaresZenodo[i][0] == dfCelsZenodoDimFrac.cel.iloc[indiceDistro] ):
        print(i,indiceDistro)
        indiceFlagelar=i
print(dfCelsZenodoDimFrac.cel.iloc[indiceDistro],curvasFlagelaresZenodo[indiceFlagelar][0])
feature=dfCelsZenodoDimFrac.dimFracDistro.iloc[indiceDistro]

cell_data=flagellar_data[indiceFlagelar]
dt = 1/90

time = np.arange(len(feature)) * dt


fig = animate_flagella_feature_curvature_plane(
    cell_data,
    feature,
    frame_duration=300,
    plane_opacity=0.4,
    smooth_sigma=1,
    peak_prominence=0.02,
    peak_distance=5,
    camera_eye=dict(x=-2, y=1.8, z=2)
)

fig.show()


In [ ]:
lim=100
indiceDistro=61
for i in range(len(curvasFlagelaresZenodo)):
    if( curvasFlagelaresZenodo[i][0] == dfCelsZenodoDimFrac.cel.iloc[indiceDistro] ):
        print(i,indiceDistro)
        indiceFlagelar=i
cell_data=flagellar_data[indiceFlagelar]

normals = []

for df in cell_data:

    centroid, e1, e2, normal = fit_plane_pca(df)

    normal = normal / np.linalg.norm(normal)

    normals.append(normal)

normals = np.array(normals)

for i in range(1, len(normals)):

    if np.dot(normals[i-1], normals[i]) < 0:
        normals[i] *= -1

nx, ny, nz = normals.T
fig=go.Figure()
fig.update_layout(
    autosize=False,
    width=1250,
    height=1000,
    scene_aspectmode='cube'
    # margin=dict(l=10, r=10, t=10, b=10, pad=10)
)
fig.add_trace(go.Scatter3d(
    x=nx[:lim],
    y=ny[:lim],
    z=nz[:lim],
    mode="lines+markers"
))
fig.add_trace(
    go.Scatter3d(
    x=[nx[0],nx[1]],
    y=[ny[0],ny[1]],
    z=[nz[0],nz[1]],
    mode='markers+lines',marker=dict(color='red')
)
)
fig.show()

n0 = normals[0]

plane_angle = np.degrees(
    np.arccos(
        np.clip(normals @ n0, -1, 1)
    )
)

fig = px.line(y=plane_angle, markers=True)
# fig.update_layout(title_text='Power spectrum of the Fractal Dimension Time-Series for '+
#                   dfCelsZenodoDimFrac.cel.iloc[indiceDistro], 
#                   title_x=0.5,font=dict(size=15))
# fig.update_layout(
#     xaxis_title="Frequency",
#     yaxis_title="Power"
# )
fig.show()
print(petrosian_fd(plane_angle))
print(higuchi_fd_fast(plane_angle))
periodicity_measure(plane_angle)

fouFD = np.fft.fft(plane_angle)
frecsFD=np.fft.fftfreq(len(plane_angle),1.0/90.0)
suplim=int(np.ceil(len(fouFD)/2))
fig = px.line(x=frecsFD[1:suplim],y=np.abs(fouFD[1:suplim]**2), markers=True)
fig.update_layout(title_text='Power spectrum of the Fractal Dimension Time-Series for ', 
                  title_x=0.5,font=dict(size=15))
fig.update_layout(
    xaxis_title="Frequency",
    yaxis_title="Power"
)
fig.show()

fouFD = np.fft.fft(plane_angle)
frequency_amplitudes = np.abs(fouFD**2)
suma=np.sum(frequency_amplitudes[1:])
maxPow=np.max(frequency_amplitudes[1:])
freqContribution=maxPow/suma
print(freqContribution)

## Sperm-1-Cap_171108_Exp9

In [ ]:
indiceDistro=116
for i in range(len(curvasFlagelaresZenodo)):
    if( curvasFlagelaresZenodo[i][0] == dfCelsZenodoDimFrac.cel.iloc[indiceDistro] ):
        print(i,indiceDistro)
        indiceFlagelar=i
print(dfCelsZenodoDimFrac.cel.iloc[indiceDistro],curvasFlagelaresZenodo[indiceFlagelar][0])
feature=dfCelsZenodoDimFrac.dimFracDistro.iloc[indiceDistro]

cell_data=flagellar_data[indiceFlagelar]
dt = 1/90

time = np.arange(len(feature)) * dt


fig = animate_flagella_feature_curvature_plane(
    cell_data,
    feature,
    frame_duration=300,
    plane_opacity=0.4,
    smooth_sigma=1,
    peak_prominence=0.02,
    peak_distance=5,
    camera_eye=dict(x=-2, y=1.8, z=2)
)

fig.show()


In [ ]:
lim=100
indiceDistro=116
for i in range(len(curvasFlagelaresZenodo)):
    if( curvasFlagelaresZenodo[i][0] == dfCelsZenodoDimFrac.cel.iloc[indiceDistro] ):
        print(i,indiceDistro)
        indiceFlagelar=i
cell_data=flagellar_data[indiceFlagelar]

normals = []

for df in cell_data:

    centroid, e1, e2, normal = fit_plane_pca(df)

    normal = normal / np.linalg.norm(normal)

    normals.append(normal)

normals = np.array(normals)

for i in range(1, len(normals)):

    if np.dot(normals[i-1], normals[i]) < 0:
        normals[i] *= -1

nx, ny, nz = normals.T
fig=go.Figure()
fig.update_layout(
    autosize=False,
    width=1250,
    height=1000,
    scene_aspectmode='cube'
    # margin=dict(l=10, r=10, t=10, b=10, pad=10)
)
fig.add_trace(go.Scatter3d(
    x=nx[:lim],
    y=ny[:lim],
    z=nz[:lim],
    mode="lines+markers"
))
fig.add_trace(
    go.Scatter3d(
    x=[nx[0],nx[1]],
    y=[ny[0],ny[1]],
    z=[nz[0],nz[1]],
    mode='markers+lines',marker=dict(color='red')
)
)
fig.show()

n0 = normals[0]

plane_angle = np.degrees(
    np.arccos(
        np.clip(normals @ n0, -1, 1)
    )
)

fig = px.line(y=plane_angle, markers=True)
# fig.update_layout(title_text='Power spectrum of the Fractal Dimension Time-Series for '+
#                   dfCelsZenodoDimFrac.cel.iloc[indiceDistro], 
#                   title_x=0.5,font=dict(size=15))
# fig.update_layout(
#     xaxis_title="Frequency",
#     yaxis_title="Power"
# )
fig.show()

print(petrosian_fd(plane_angle))
print(higuchi_fd_fast(plane_angle))
print(periodicity_measure(plane_angle))

# fouFD = np.fft.fft(plane_angle)
# frecsFD=np.fft.fftfreq(len(plane_angle),1.0/90.0)
# suplim=int(np.ceil(len(fouFD)/2))
# fig = px.line(x=frecsFD[1:suplim],y=np.abs(fouFD[1:suplim]**2), markers=True)
# fig.update_layout(title_text='Power spectrum of the Fractal Dimension Time-Series for ', 
#                   title_x=0.5,font=dict(size=15))
# fig.update_layout(
#     xaxis_title="Frequency",
#     yaxis_title="Power"
# )
# fig.show()

fouFD = np.fft.fft(plane_angle)
frequency_amplitudes = np.abs(fouFD**2)
suma=np.sum(frequency_amplitudes[1:])
maxPow=np.max(frequency_amplitudes[1:])
freqContribution=maxPow/suma
print(freqContribution)

## Sperm-12-NoCap_210730_Exp18

In [ ]:
indiceDistro=39
for i in range(len(curvasFlagelaresZenodo)):
    if( curvasFlagelaresZenodo[i][0] == dfCelsZenodoDimFrac.cel.iloc[indiceDistro] ):
        print(i)
        indiceFlagelar=i
print(dfCelsZenodoDimFrac.cel.iloc[indiceDistro],curvasFlagelaresZenodo[indiceFlagelar][0])
feature=dfCelsZenodoDimFrac.dimFracDistro.iloc[indiceDistro]
cell_data=flagellar_data[indiceFlagelar]
dt = 1/90

time = np.arange(len(feature)) * dt

fig = animate_flagella_with_feature_vertical_curvature(
    cell_data,
    feature,
    time=time,
    feature_name="Fractal dimension of "+dfCelsZenodoDimFrac.cel.iloc[indiceDistro],
    # flagellarTitle=curvasFlagelaresZenodo[indiceFlagelar][0],
    frame_duration=150,
    peak_prominence=0.02,
    peak_distance=8,
    smooth_sigma=1
)

fig.show()

In [ ]:
lim=100
indiceDistro=39
for i in range(len(curvasFlagelaresZenodo)):
    if( curvasFlagelaresZenodo[i][0] == dfCelsZenodoDimFrac.cel.iloc[indiceDistro] ):
        print(i,indiceDistro)
        indiceFlagelar=i
cell_data=flagellar_data[indiceFlagelar]

normals = []

for df in cell_data:

    centroid, e1, e2, normal = fit_plane_pca(df)

    normal = normal / np.linalg.norm(normal)

    normals.append(normal)

normals = np.array(normals)

for i in range(1, len(normals)):

    if np.dot(normals[i-1], normals[i]) < 0:
        normals[i] *= -1

nx, ny, nz = normals.T
fig=go.Figure()
fig.update_layout(
    autosize=False,
    width=1250,
    height=1000,
    scene_aspectmode='cube'
    # margin=dict(l=10, r=10, t=10, b=10, pad=10)
)
fig.add_trace(go.Scatter3d(
    x=nx[:lim],
    y=ny[:lim],
    z=nz[:lim],
    mode="lines+markers"
))
fig.add_trace(
    go.Scatter3d(
    x=[nx[0],nx[1]],
    y=[ny[0],ny[1]],
    z=[nz[0],nz[1]],
    mode='markers+lines',marker=dict(color='red')
)
)
fig.show()

n0 = normals[0]

plane_angle = np.degrees(
    np.arccos(
        np.clip(normals @ n0, -1, 1)
    )
)

fig = px.line(y=plane_angle, markers=True)
# fig.update_layout(title_text='Power spectrum of the Fractal Dimension Time-Series for '+
#                   dfCelsZenodoDimFrac.cel.iloc[indiceDistro], 
#                   title_x=0.5,font=dict(size=15))
# fig.update_layout(
#     xaxis_title="Frequency",
#     yaxis_title="Power"
# )
fig.show()
print(petrosian_fd(plane_angle))
print(higuchi_fd_fast(plane_angle))
print(periodicity_measure(plane_angle))

fouFD = np.fft.fft(plane_angle)
frecsFD=np.fft.fftfreq(len(plane_angle),1.0/90.0)
suplim=int(np.ceil(len(fouFD)/2))
fig = px.line(x=frecsFD[1:suplim],y=np.abs(fouFD[1:suplim]**2), markers=True)
fig.update_layout(title_text='Power spectrum of the Fractal Dimension Time-Series for ', 
                  title_x=0.5,font=dict(size=15))
fig.update_layout(
    xaxis_title="Frequency",
    yaxis_title="Power"
)
fig.show()

# Torsion and Curvature

In [ ]:
df=cell_data[12]
torsion=compute_torsion(
            df,
            "x",
            "y",
            "z",
            smooth_sigma=2
        )

In [ ]:
# indiceDistro=116
# print(np.max(dfCelsZenodoDimFrac.dimFracDistro.iloc[indiceDistro])-np.min(dfCelsZenodoDimFrac.dimFracDistro.iloc[indiceDistro]))
fig = px.line(y=torsion, markers=True)
fig.update_layout(title_text='torsion', 
                  title_x=0.5,font=dict(size=15))
fig.show()

# Higuchi and Petrosian FD

In [ ]:
ncels=dfCelsZenodoDimFrac.shape[0]
petrosianFDList=[]
for i in range(ncels):
    nom=dfCelsZenodoDimFrac.cel.iloc[i]
    pfd=petrosian_fd(dfCelsZenodoDimFrac.dimFracDistro.iloc[i][:300])
    if( 'No' in dfCelsZenodoDimFrac.cel.iloc[i] ):
        cond='NoCap'
    else:
        cond='Cap'
    petrosianFDList.append([nom,cond,pfd])

In [ ]:
fdPetrosianFD=pd.DataFrame(petrosianFDList, columns=['cel','Condition','PetrosianFD'])

In [ ]:
fig=px.violin(fdPetrosianFD, x='PetrosianFD', color='Condition', box=True, points='all', #log_x=True,
             category_orders={"Condition":["NoCap","Cap"]})
fig.update_traces(meanline_visible=True)
fig.update_layout(title_text="Distribution of the Petrosian Fractal Dimmension of each Cell's KFD Time-series", 
# fig.update_layout(title_text='Distribución de la frecuencia principal para la serie de tiempo de cada célula', 
                 title_x=0.5,font=dict(size=20),xaxis_title="Petrosian FD")
fig.show()

In [ ]:
# ncels=dfCelsZenodoDimFrac.shape[0]
# higuchiFDList=[]
# for i in range(ncels):
#     nom=dfCelsZenodoDimFrac.cel.iloc[i]
#     hfd=higuchi_fd_fast(dfCelsZenodoDimFrac.dimFracDistro.iloc[i][:300])
#     if( 'No' in dfCelsZenodoDimFrac.cel.iloc[i] ):
#         cond='NoCap'
#     else:
#         cond='Cap'
#     higuchiFDList.append([nom,cond,hfd])

In [ ]:
# fdHiguchiFD=pd.DataFrame(higuchiFDList, columns=['cel','Condition','HiguchiFD'])

In [ ]:
# fig=px.violin(fdHiguchiFD, x='HiguchiFD', color='Condition', box=True, points='all', #log_x=True,
#              category_orders={"Condition":["NoCap","Cap"]})
# fig.update_traces(meanline_visible=True)
# fig.update_layout(title_text="Distribution of the Higuchi Fractal Dimmension of each Cell's KFD Time-series", 
# # fig.update_layout(title_text='Distribución de la frecuencia principal para la serie de tiempo de cada célula', 
#                  title_x=0.5,font=dict(size=20),xaxis_title="Higuchi FD")
# fig.show()

In [ ]:
fdPetrosianFD[fdPetrosianFD['PetrosianFD']>0.91].sort_values(by=['PetrosianFD'],ascending=False).head(10)

In [ ]:
fdPetrosianFD[fdPetrosianFD['PetrosianFD']<1.02].sort_values(by=['PetrosianFD'],ascending=False).head(10)

In [ ]:
fdPetrosianFD[fdPetrosianFD['PetrosianFD']<1].sort_values(by=['PetrosianFD'],ascending=False).head(20)

In [ ]:
fdPetrosianFD[fdPetrosianFD['PetrosianFD']<0.6].sort_values(by=['PetrosianFD'],ascending=False)

In [ ]:
print(fdPetrosianFD.iloc[62])
print(fdPetrosianFD.iloc[2])
print(fdPetrosianFD.iloc[13])
fig=go.Figure()
fig.add_trace(go.Scatter(y=dfCelsZenodoDimFrac.dimFracDistro.iloc[126], mode='lines+markers', name=dfCelsZenodoDimFrac.cel.iloc[126]))
fig.add_trace(go.Scatter(y=dfCelsZenodoDimFrac.dimFracDistro.iloc[116], mode='lines+markers', name=dfCelsZenodoDimFrac.cel.iloc[116]))
fig.add_trace(go.Scatter(y=dfCelsZenodoDimFrac.dimFracDistro.iloc[62], mode='lines+markers', name=dfCelsZenodoDimFrac.cel.iloc[62]))
fig.update_layout(title="5 Sine waves", xaxis_title="X Axis", yaxis_title="Y Axis")
fig.show()

In [ ]:
freqsComp(62,2), freqsComp(2,13)

In [ ]:
indiceDistro=2
fouFD = np.fft.fft(dfCelsZenodoDimFrac.dimFracDistro.iloc[indiceDistro])
frecsFD=np.fft.fftfreq(len(dfCelsZenodoDimFrac.dimFracDistro.iloc[indiceDistro]),1.0/90.0)
suplim=int(np.ceil(len(fouFD)/2))
fig = px.line(x=frecsFD[1:suplim],y=np.abs(fouFD[1:suplim]**2), markers=True)
fig.update_layout(title_text='Power spectrum of the Fractal Dimension Time-Series for '+
                  dfCelsZenodoDimFrac.cel.iloc[indiceDistro], 
                  title_x=0.5,font=dict(size=15))
fig.update_layout(
    xaxis_title="Frequency",
    yaxis_title="Power"
)
fig.show()

In [ ]:
indiceDistro=13
fouFD = np.fft.fft(dfCelsZenodoDimFrac.dimFracDistro.iloc[indiceDistro])
frecsFD=np.fft.fftfreq(len(dfCelsZenodoDimFrac.dimFracDistro.iloc[indiceDistro]),1.0/90.0)
suplim=int(np.ceil(len(fouFD)/2))
fig = px.line(x=frecsFD[1:suplim],y=np.abs(fouFD[1:suplim]**2), markers=True)
fig.update_layout(title_text='Power spectrum of the Fractal Dimension Time-Series for '+
                  dfCelsZenodoDimFrac.cel.iloc[indiceDistro], 
                  title_x=0.5,font=dict(size=15))
fig.update_layout(
    xaxis_title="Frequency",
    yaxis_title="Power"
)
fig.show()

In [ ]:
print(fdPetrosianFD.iloc[39])
print(fdPetrosianFD.iloc[105])
print(fdPetrosianFD.iloc[61])
fig=go.Figure()
fig.add_trace(go.Scatter(y=dfCelsZenodoDimFrac.dimFracDistro.iloc[126], mode='lines+markers', name=dfCelsZenodoDimFrac.cel.iloc[126]))
fig.add_trace(go.Scatter(y=dfCelsZenodoDimFrac.dimFracDistro.iloc[116], mode='lines+markers', name=dfCelsZenodoDimFrac.cel.iloc[116]))
fig.add_trace(go.Scatter(y=dfCelsZenodoDimFrac.dimFracDistro.iloc[62], mode='lines+markers', name=dfCelsZenodoDimFrac.cel.iloc[62]))
fig.update_layout(title="5 Sine waves", xaxis_title="X Axis", yaxis_title="Y Axis")
fig.show()

In [ ]:
freqsComp(105, 61)

In [ ]:
freqsComp(39,105)

In [ ]:
print(fdPetrosianFD.iloc[62])
print(fdPetrosianFD.iloc[39])
print(fdPetrosianFD.iloc[116])
fig=go.Figure()
fig.add_trace(go.Scatter(y=dfCelsZenodoDimFrac.dimFracDistro.iloc[126], mode='lines+markers', name=dfCelsZenodoDimFrac.cel.iloc[126]))
fig.add_trace(go.Scatter(y=dfCelsZenodoDimFrac.dimFracDistro.iloc[116], mode='lines+markers', name=dfCelsZenodoDimFrac.cel.iloc[116]))
fig.add_trace(go.Scatter(y=dfCelsZenodoDimFrac.dimFracDistro.iloc[62], mode='lines+markers', name=dfCelsZenodoDimFrac.cel.iloc[62]))
fig.update_layout(title="5 Sine waves", xaxis_title="X Axis", yaxis_title="Y Axis")
fig.show()

In [ ]:
indiceDistro=116
for i in range(len(curvasFlagelaresZenodo)):
    if( curvasFlagelaresZenodo[i][0] == dfCelsZenodoDimFrac.cel.iloc[indiceDistro] ):
        print(i)
        indiceFlagelar=i
print(dfCelsZenodoDimFrac.cel.iloc[indiceDistro],curvasFlagelaresZenodo[indiceFlagelar][0])
feature=dfCelsZenodoDimFrac.dimFracDistro.iloc[indiceDistro]
cell_data=flagellar_data[indiceFlagelar]
dt = 1/90

time = np.arange(len(feature)) * dt

fig = animate_flagella_with_feature_vertical_curvature(
    cell_data,
    feature,
    time=time,
    feature_name="Fractal dimension of "+dfCelsZenodoDimFrac.cel.iloc[indiceDistro],
    # flagellarTitle=curvasFlagelaresZenodo[indiceFlagelar][0],
    frame_duration=150,
    peak_prominence=0.02,
    peak_distance=5,
    smooth_sigma=1
)

fig.show()

In [ ]:
indiceDistro=126
for i in range(len(curvasFlagelaresZenodo)):
    if( curvasFlagelaresZenodo[i][0] == dfCelsZenodoDimFrac.cel.iloc[indiceDistro] ):
        print(i)
        indiceFlagelar=i
print(dfCelsZenodoDimFrac.cel.iloc[indiceDistro],curvasFlagelaresZenodo[indiceFlagelar][0])
feature=dfCelsZenodoDimFrac.dimFracDistro.iloc[indiceDistro]
cell_data=flagellar_data[indiceFlagelar]
dt = 1/90

time = np.arange(len(feature)) * dt

fig = animate_flagella_with_feature_vertical_curvature(
    cell_data,
    feature,
    time=time,
    feature_name="Fractal dimension of "+dfCelsZenodoDimFrac.cel.iloc[indiceDistro],
    # flagellarTitle=curvasFlagelaresZenodo[indiceFlagelar][0],
    frame_duration=150,
    peak_prominence=0.02,
    peak_distance=5,
    smooth_sigma=1
)

fig.show()

In [ ]:
indiceDistro=62
for i in range(len(curvasFlagelaresZenodo)):
    if( curvasFlagelaresZenodo[i][0] == dfCelsZenodoDimFrac.cel.iloc[indiceDistro] ):
        print(i)
        indiceFlagelar=i
print(dfCelsZenodoDimFrac.cel.iloc[indiceDistro],curvasFlagelaresZenodo[indiceFlagelar][0])
feature=dfCelsZenodoDimFrac.dimFracDistro.iloc[indiceDistro]
cell_data=flagellar_data[indiceFlagelar]
dt = 1/90

time = np.arange(len(feature)) * dt

fig = animate_flagella_with_feature_vertical_curvature(
    cell_data,
    feature,
    time=time,
    feature_name="Fractal dimension of "+dfCelsZenodoDimFrac.cel.iloc[indiceDistro],
    # flagellarTitle=curvasFlagelaresZenodo[indiceFlagelar][0],
    frame_duration=150,
    peak_prominence=0.02,
    peak_distance=5,
    smooth_sigma=1
)

fig.show()

In [ ]:
# Sinusoidal wave generation
# Parameters
n_samples = 2000
window_size = 100

X = []

for _ in range(n_samples):

    # Random sine parameters
    freq = np.random.uniform(0.5, 3.0)
    phase = np.random.uniform(0, 2*np.pi)
    amplitude = np.random.uniform(0.5, 1.5)

    t = np.linspace(0, 1, window_size)

    signal = amplitude * np.sin(2*np.pi*freq*t + phase)

    # Optional noise
    signal += 0.1 * np.random.randn(window_size)

    X.append(signal)

X = np.array(X)

print(X.shape)

In [ ]:
fig=go.Figure()
for i in range(5):
    fig.add_trace(go.Scatter(y=X[i], mode='lines+markers', name=i))

fig.update_layout(title="5 Sine waves", xaxis_title="X Axis", yaxis_title="Y Axis")
fig.show()

In [ ]:
for i in range(5):
    print(higuchi_fd_fast(X[i]))

In [ ]:
indiceDistro=62
for i in range(len(curvasFlagelaresZenodo)):
    if( curvasFlagelaresZenodo[i][0] == dfCelsZenodoDimFrac.cel.iloc[indiceDistro] ):
        print(i)
        indiceFlagelar=i
print(dfCelsZenodoDimFrac.cel.iloc[indiceDistro],curvasFlagelaresZenodo[indiceFlagelar][0])
feature=dfCelsZenodoDimFrac.dimFracDistro.iloc[indiceDistro]

cell_data=flagellar_data[indiceFlagelar]
dt = 1/90

time = np.arange(len(feature)) * dt

fig = animate_flagella_feature_curvature_plane(
    cell_data,
    feature,
    frame_duration=200,
    plane_opacity=0.4,
    smooth_sigma=1,
    peak_prominence=0.02,
    peak_distance=5,

)

fig.show()


In [ ]:
indiceDistro=116
for i in range(len(curvasFlagelaresZenodo)):
    if( curvasFlagelaresZenodo[i][0] == dfCelsZenodoDimFrac.cel.iloc[indiceDistro] ):
        print(i)
        indiceFlagelar=i
print(dfCelsZenodoDimFrac.cel.iloc[indiceDistro],curvasFlagelaresZenodo[indiceFlagelar][0])
feature=dfCelsZenodoDimFrac.dimFracDistro.iloc[indiceDistro]

cell_data=flagellar_data[indiceFlagelar]
dt = 1/90

time = np.arange(len(feature)) * dt


fig = animate_flagella_feature_curvature_plane(
    cell_data,
    feature,
    frame_duration=300,
    plane_opacity=0.4,
    smooth_sigma=1,
    peak_prominence=0.02,
    peak_distance=5,
)

fig.show()


In [ ]:
fig=go.Figure()
fig.add_trace(go.Scatter(y=dfCelsZenodoDimFrac.dimFracDistro.iloc[34], mode='lines+markers', name=dfCelsZenodoDimFrac.cel.iloc[34]))
fig.add_trace(go.Scatter(y=dfCelsZenodoDimFrac.dimFracDistro.iloc[110], mode='lines+markers', name=dfCelsZenodoDimFrac.cel.iloc[110]))
fig.add_trace(go.Scatter(y=dfCelsZenodoDimFrac.dimFracDistro.iloc[45], mode='lines+markers', name=dfCelsZenodoDimFrac.cel.iloc[45]))
fig.update_layout(title="5 Sine waves", xaxis_title="X Axis", yaxis_title="Y Axis")
fig.show()

In [ ]:
dfZenodoFDMaxAmpDistNor[dfZenodoFDMaxAmpDistNor['maxAmp']< 0.1 ].sort_values(by=['maxAmp']).head()

In [ ]:
higuchi_fd_fast(dfCelsZenodoDimFrac.dimFracDistro.iloc[74]),higuchi_fd_fast(dfCelsZenodoDimFrac.dimFracDistro.iloc[75]),higuchi_fd_fast(dfCelsZenodoDimFrac.dimFracDistro.iloc[77])

In [ ]:
higuchi_fd_fast(dfCelsZenodoDimFrac.dimFracDistro.iloc[116])

In [ ]:
fig=go.Figure()
fig.add_trace(go.Scatter(y=dfCelsZenodoDimFrac.dimFracDistro.iloc[74], mode='lines+markers', name=dfCelsZenodoDimFrac.cel.iloc[74]))
fig.add_trace(go.Scatter(y=dfCelsZenodoDimFrac.dimFracDistro.iloc[75], mode='lines+markers', name=dfCelsZenodoDimFrac.cel.iloc[75]))
fig.add_trace(go.Scatter(y=dfCelsZenodoDimFrac.dimFracDistro.iloc[77], mode='lines+markers', name=dfCelsZenodoDimFrac.cel.iloc[77]))
fig.update_layout(title="5 Sine waves", xaxis_title="X Axis", yaxis_title="Y Axis")
fig.show()

In [ ]:
indiceDistro=74
for i in range(len(curvasFlagelaresZenodo)):
    if( curvasFlagelaresZenodo[i][0] == dfCelsZenodoDimFrac.cel.iloc[indiceDistro] ):
        print(i)
        indiceFlagelar=i
print(dfCelsZenodoDimFrac.cel.iloc[indiceDistro],curvasFlagelaresZenodo[indiceFlagelar][0])
feature=dfCelsZenodoDimFrac.dimFracDistro.iloc[indiceDistro]

cell_data=flagellar_data[indiceFlagelar]
dt = 1/90

time = np.arange(len(feature)) * dt


fig = animate_flagella_feature_curvature_plane(
    cell_data,
    feature,
    frame_duration=300,
    plane_opacity=0.4,
    smooth_sigma=1,
    peak_prominence=0.02,
    peak_distance=5,
)

fig.show()


In [ ]:
len(cell_data), len(feature)

In [ ]:
indiceDistro=34
for i in range(len(curvasFlagelaresZenodo)):
    if( curvasFlagelaresZenodo[i][0] == dfCelsZenodoDimFrac.cel.iloc[indiceDistro] ):
        print(i)
        indiceFlagelar=i
print(dfCelsZenodoDimFrac.cel.iloc[indiceDistro],curvasFlagelaresZenodo[indiceFlagelar][0])
feature=dfCelsZenodoDimFrac.dimFracDistro.iloc[indiceDistro]

cell_data=flagellar_data[indiceFlagelar]
# dt = 1/90

# time = np.arange(len(feature)) * dt


fig = animate_flagella_feature_curvature_plane(
    cell_data[:len(feature)],
    feature,
    frame_duration=300,
    plane_opacity=0.4,
    smooth_sigma=1,
    peak_prominence=0.02,
    peak_distance=5,
)

fig.show()


## Weierstrass Function

In [ ]:
def weierstrass_vectorized(x, a=0.5, b=7, n_terms=50):
    x = np.asarray(x)

    n = np.arange(n_terms)[:, None]

    terms = (a ** n) * np.cos((b ** n) * np.pi * x)

    return terms.sum(axis=0)

In [ ]:
x = np.linspace(0, 0.01, 5000)
a1=0.5
b1=11
y1 = weierstrass_vectorized(x, a=a1, b=b1, n_terms=80)
print('HFD(y1)=',higuchi_fd_fast(y1))
a2=0.75
b2=11
y2 = weierstrass_vectorized(x, a=a2, b=b2, n_terms=80)
print('HFD(y2)=',higuchi_fd_fast(y2))

fig=go.Figure()
fig.add_trace(go.Scatter(y=y1, mode='lines+markers', name='y1: a='+str(a1)+', b='+str(b1)))
fig.add_trace(go.Scatter(y=y2, mode='lines+markers', name='y2: a='+str(a2)+', b='+str(b2)))
fig.update_layout(title="5 Sine waves", xaxis_title="X Axis", yaxis_title="Y Axis")
fig.show()